# 🎙️ Clonador de voz · XTTS-v2 y Chatterbox en Google Colab

Convierte cualquier texto en audio **con tu propia voz**, a partir de una grabación tuya en **Google Drive** (o subida desde tu computadora). Todo se procesa en la **GPU** de Colab.

### Cómo usarlo
1. Comprueba que el entorno tenga GPU: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**. El cuaderno ya viene configurado así.
2. En la celda **1** elige el modelo: **XTTS-v2** (el más parecido; uso no comercial) o **Chatterbox** (uso comercial).
3. Completa la celda **4** (tu audio en Drive) y ejecuta las celdas **1 → 6** en orden, o todas a la vez con **Ctrl + F9**. La primera vez, Colab te pedirá permiso para acceder a tu Drive. El texto se escribe en el cuadro que aparece en la celda **5**.
4. Para otro audio con la misma voz, cambia el texto y ejecuta **solo la celda 6**. Si pides varias **tomas**, la celda 6 las ordena por calidad y parecido, y guardas la que prefieras con la **7**.
5. La celda 4 guarda tu voz como **perfil** en Drive: en otra sesión, elige «Perfil guardado» y estará lista en segundos.

Cada celda muestra solo sus parámetros. Para ver o editar su código, haz **doble clic sobre el formulario** de la celda (o elige *Mostrar código* en su menú **⋮**).

### Qué más puede hacer
- **Diálogos** con varias voces: `[voz: Ana Florence]¿Vienes mañana?[/voz]` (voces incluidas de XTTS-v2 o tus perfiles guardados).
- **Otros idiomas** con tu misma voz: `[en]Where is the library?[/en]`.
- **Lotes**: un audio por bloque separado con `---` (capítulos, lecciones), cada uno con su nombre.
- **Control de calidad** automático (ritmo, silencios y, si quieres, Whisper), **subtítulos `.srt`**, **música de fondo** que baja sola cuando hablas y volumen parejo en LUFS.
- **Entrenar XTTS-v2 con tu voz** para un parecido todavía mayor, con el cuaderno aparte [Entrenar XTTS con tu voz](https://colab.research.google.com/github/JoelPasapera/clonador-voz/blob/main/Entrenar_XTTS_con_tu_voz.ipynb).

### Para un buen parecido
- Graba **entre 30 y 60 segundos** hablando con naturalidad, sin eco, sin música y con una sola voz. Sirven WAV, MP3, M4A, OGG/OPUS (notas de voz de WhatsApp) e incluso videos.
- El modelo imita el tono y el ritmo de la grabación: si quieres voz de narrador, graba como narrador.
- Si el modelo lee mal una sigla o un nombre, corrígelo en **Pronunciación** (celda 5), p. ej. `DNI = de ene i`.

> **Licencias y uso responsable.** **XTTS-v2** se distribuye con la [Coqui Public Model License](https://tts-hub.github.io/cpml), que solo permite **uso no comercial** del modelo y de los audios que genera: el cuaderno **asume ese uso por defecto** y acepta la licencia en tu nombre (casilla `uso_no_comercial` de la celda 3, ya marcada); desmárcala si vas a darle a los audios un uso comercial. **Chatterbox** tiene licencia MIT, que permite el uso comercial; sus audios llevan una marca de agua inaudible que los identifica como generados por IA. Con cualquiera de los dos, clona únicamente tu voz o la de alguien que te haya dado su permiso.

In [ ]:
# @title 1 · Preparar el entorno (GPU y librerías) { display-mode: "form" }
# @markdown Elige el **modelo de voz**. **XTTS-v2** da el mejor parecido y permite entrenar con tu voz (cuaderno «Entrenar XTTS con tu voz»), pero solo para **uso no comercial**. **Chatterbox** permite **uso comercial** (licencia MIT; sus audios llevan una marca de agua inaudible).
modelo = "XTTS-v2 (uso no comercial)"  # @param ["XTTS-v2 (uso no comercial)", "Chatterbox Multilingual (uso comercial)"]
# @markdown *La primera vez tarda 1–3 minutos (Chatterbox, unos minutos más); si la vuelves a ejecutar, no reinstala nada. Para cambiar de modelo: «Entorno de ejecución → Desconectar y eliminar entorno de ejecución» y empieza otra vez.*

"""Celda 1 · Entorno: verifica la GPU, instala lo que falte (con uv, mucho más rápido que pip) y reinicia la sesión
solo si es imprescindible. También deja `instalar()` para las celdas que necesiten algo más (Whisper)."""

import importlib
import importlib.metadata as metadata
import os
import re
import shutil
import signal
import subprocess
import sys
import time

VERSION_COQUI_TTS = "0.28.0"  # versiones con las que se verificó este cuaderno
VERSION_CHATTERBOX = "0.1.7"
VERSION_FASTER_WHISPER = "1.2.1"
TORCHVISION_DE_CHATTERBOX = "0.21.0"  # la que corresponde al torch 2.6.0 que fija Chatterbox
MODELOS = {"XTTS-v2 (uso no comercial)": "xtts", "Chatterbox Multilingual (uso comercial)": "chatterbox"}
PAQUETES_EXTRA: list[str] = []  # p. ej. ["transformers<5.19"] si una actualización de Colab rompe la instalación
MODULOS_SENSIBLES = ("numpy", "scipy", "torch", "torchaudio", "transformers", "tokenizers", "huggingface_hub")


class ErrorUsuario(Exception):
    """Problema que el usuario puede corregir: se muestra como un mensaje, sin traceback."""


def confirmar_licencia(no_comercial: bool) -> None:
    """Aplica el caso de uso declarado en la casilla de XTTS-v2 y lo deja a la vista en cada ejecución (transparencia).
    La usan la celda 3 del clonador y la del cuaderno de entrenamiento."""
    if not no_comercial:
        os.environ.pop("COQUI_TOS_AGREED", None)
        raise ErrorUsuario(
            "Desmarcaste «uso_no_comercial»: la licencia de XTTS-v2 (CPML) no permite el uso comercial, así que "
            "XTTS-v2 no se usa. Si fue sin querer, vuelve a marcar la casilla y ejecuta esta celda otra vez; para "
            "uso comercial, elige Chatterbox en la celda 1 del clonador."
        )
    os.environ["COQUI_TOS_AGREED"] = "1"  # forma prevista por coqui-tts para aceptar la CPML sin preguntas interactivas
    print("ℹ️ Licencia: CPML aceptada para uso no comercial (casilla «uso_no_comercial» de esta celda).")


def registrar_errores_amigables() -> None:
    """Muestra cada ErrorUsuario en una sola línea roja en lugar de un traceback.

    IPython ignora lo que devuelve el manejador al ejecutar una celda, por eso imprime el mensaje;
    la celda igual queda marcada con error y «Ejecutar todas» se detiene ahí.
    """
    try:
        shell = get_ipython()  # noqa: F821 (la define IPython/Colab)
    except NameError:
        return
    if shell is None:
        return

    def mostrar(_shell, _tipo, valor, _traza, tb_offset=None) -> None:
        print(f"\x1b[1;31m❌ {valor}\x1b[0m", file=sys.stderr)

    shell.set_custom_exc((ErrorUsuario,), mostrar)


def version_instalada(paquete: str) -> str | None:
    """Versión instalada de un paquete, o None si no está."""
    try:
        return metadata.version(paquete)
    except metadata.PackageNotFoundError:
        return None


def version_mayor_menor(version: str) -> tuple[int, int]:
    """'2.10.0+cu126' → (2, 10)."""
    coincidencia = re.match(r"(\d+)\.(\d+)", version)
    if coincidencia is None:
        raise ValueError(f"Versión no reconocida: {version}")
    return int(coincidencia.group(1)), int(coincidencia.group(2))


def verificar_gpu() -> str:
    """Nombre y memoria de la GPU; error claro si el entorno no tiene una."""
    if shutil.which("nvidia-smi") is None:
        raise ErrorUsuario(
            "Este entorno no tiene GPU. Ve a «Entorno de ejecución → Cambiar tipo de entorno de ejecución», "
            "elige «T4 GPU», guarda y ejecuta otra vez desde la celda 1."
        )
    consulta = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
        check=False,
    )
    lineas = consulta.stdout.strip().splitlines()
    if consulta.returncode != 0 or not lineas:
        raise ErrorUsuario("La GPU no responde. Usa «Entorno de ejecución → Reiniciar sesión» y vuelve a intentarlo.")
    return lineas[0].strip()


def asegurar_ffmpeg() -> None:
    """FFmpeg convierte tu audio (MP3, M4A, OPUS, video…). Colab ya lo trae; si faltara, se instala."""
    if shutil.which("ffmpeg") is None:
        subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg", shell=True, check=True, capture_output=True)


def especificacion_torchcodec(torch: tuple[int, int]) -> str | None:
    """torchcodec compatible con la versión de torch, según la tabla oficial de torchcodec.

    coqui-tts lo exige desde torch 2.9, porque torchaudio le delega la lectura de audio.
    """
    if torch < (2, 9):
        return None
    tabla = {(2, 9): "torchcodec==0.9.*", (2, 10): "torchcodec==0.10.*"}
    return tabla.get(torch, "torchcodec>=0.12")  # desde la 0.12 sirve para cualquier torch >= 2.11


def paquetes_pendientes(motor: str) -> list[str]:
    """Lo que falta instalar para el modelo elegido, partiendo del torch que ya trae Colab."""
    version_torch = version_instalada("torch")
    if version_torch is None:
        raise ErrorUsuario("Este entorno no tiene PyTorch. Usa un entorno estándar de Colab con GPU.")
    if motor == "chatterbox":  # trae su propio torch (2.6.0); torchvision debe ir a juego o falla al importarse
        if version_instalada("chatterbox-tts") == VERSION_CHATTERBOX:
            return list(PAQUETES_EXTRA)
        return [f"chatterbox-tts=={VERSION_CHATTERBOX}", f"torchvision=={TORCHVISION_DE_CHATTERBOX}", *PAQUETES_EXTRA]
    pendientes = []
    if version_instalada("coqui-tts") != VERSION_COQUI_TTS:
        pendientes.append(f"coqui-tts=={VERSION_COQUI_TTS}")
    if version_instalada("torchaudio") is None:
        pendientes.append(f"torchaudio=={version_torch}")  # misma versión y misma variante CUDA que torch
    codec = especificacion_torchcodec(version_mayor_menor(version_torch))
    if codec is not None and version_instalada("torchcodec") is None:
        pendientes.append(codec)
    return pendientes + list(PAQUETES_EXTRA)


def indices_pytorch(especificaciones: list[str]) -> list[str]:
    """Índices de PyTorch para las variantes CUDA pedidas (p. ej. «+cu126»), que no están publicadas en PyPI."""
    variantes = sorted({e.split("+", 1)[1] for e in especificaciones if "+" in e})
    return [f"https://download.pytorch.org/whl/{variante}" for variante in variantes]


def asegurar_uv() -> str | None:
    """Ruta de uv (instalador en Rust, varias veces más rápido que pip); si no está, se instala en segundos."""
    if shutil.which("uv") is None:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", "uv"],
            capture_output=True,
            check=False,
        )
    return shutil.which("uv")


def comandos_de_instalacion(especificaciones: list[str], indices_extra: list[str]) -> list[list[str]]:
    """Comandos a probar en orden: uv sobre el Python de esta sesión y, si falla, pip."""
    comandos = []
    uv = asegurar_uv()
    if uv is not None:
        comando = [uv, "pip", "install", "--quiet", "--python", sys.executable, "--system", "--break-system-packages"]
        for indice in indices_extra:
            comando += ["--extra-index-url", indice]
        if indices_extra:
            comando += ["--index-strategy", "unsafe-best-match"]  # combinar índices como lo hace pip
        comandos.append(comando + especificaciones)
    comando = [sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check"]
    for indice in indices_extra:
        comando += ["--extra-index-url", indice]
    comandos.append(comando + especificaciones)
    return comandos


def instalar(especificaciones: list[str], indices_extra: list[str] | None = None) -> None:
    """Instala paquetes con uv (o pip si uv falla); si nada funciona, muestra el final del registro."""
    resultado = None
    for comando in comandos_de_instalacion(especificaciones, list(indices_extra or [])):
        resultado = subprocess.run(comando, capture_output=True, text=True, check=False)
        if resultado.returncode == 0:
            importlib.invalidate_caches()  # que esta misma sesión vea los paquetes recién instalados
            return
    print(resultado.stdout[-2000:], resultado.stderr[-4000:], sep="\n")
    raise RuntimeError("No se pudo instalar: " + " ".join(especificaciones))


def instalar_si_falta(paquete: str, especificacion: str) -> None:
    """Para dependencias opcionales que solo algunas celdas usan (p. ej. faster-whisper)."""
    if version_instalada(paquete) is None:
        print(f"📦 Instalando {especificacion}…")
        instalar([especificacion])


def modulos_desactualizados() -> list[str]:
    """Librerías ya cargadas en memoria cuya versión en disco acaba de cambiar."""
    cambios = []
    for nombre in MODULOS_SENSIBLES:
        cargada = getattr(sys.modules.get(nombre), "__version__", None)
        instalada = version_instalada(nombre)
        if cargada and instalada and cargada.split("+")[0] != instalada.split("+")[0]:
            cambios.append(f"{nombre} {cargada} → {instalada}")
    return cambios


def reiniciar_sesion(cambios: list[str]) -> None:
    """Reinicia el kernel para que cargue las versiones nuevas (lo instalado se conserva)."""
    print("🔄 Se actualizaron librerías que ya estaban en memoria:", "; ".join(cambios))
    print("   Reiniciando la sesión… Cuando se reconecte, ejecuta otra vez desde la celda 1 (será inmediata).")
    sys.stdout.flush()
    os.kill(os.getpid(), signal.SIGKILL)


def preparar_entorno(eleccion: str) -> str:
    """Devuelve el motor elegido («xtts» o «chatterbox») con todo listo para usarlo."""
    inicio = time.perf_counter()
    registrar_errores_amigables()
    motor = MODELOS[eleccion]
    if motor == "xtts" and version_instalada("chatterbox-tts") is not None:
        raise ErrorUsuario(
            "Este entorno ya tiene instalado Chatterbox, que cambió las versiones de PyTorch y transformers. Para "
            "usar XTTS-v2: «Entorno de ejecución → Desconectar y eliminar entorno de ejecución» y empieza otra vez."
        )
    gpu = verificar_gpu()
    asegurar_ffmpeg()
    pendientes = paquetes_pendientes(motor)
    if pendientes:
        print("📦 Instalando:", ", ".join(pendientes), "(unos minutos)…")
        instalar(pendientes, indices_pytorch(pendientes))
    cambios = modulos_desactualizados()
    if cambios:
        reiniciar_sesion(cambios)
    paquete = "chatterbox-tts" if motor == "chatterbox" else "coqui-tts"
    print(f"✅ Entorno listo en {time.perf_counter() - inicio:.0f} s · modelo: {eleccion}")
    print(
        f"   GPU: {gpu} · Python {sys.version.split()[0]} · "
        f"torch {version_instalada('torch')} · {paquete} {version_instalada(paquete)}"
    )
    return motor


# ─── Ejecución ───
MOTOR_ELEGIDO = preparar_entorno(modelo)

In [ ]:
# @title 2a · Funciones: Drive y archivos { display-mode: "form" }
# @markdown Acceso a tu Google Drive, búsqueda de archivos por nombre y nombres de salida. No tiene parámetros: solo ejecútala.

"""Celda 2a · Archivos: Google Drive (montaje, rutas y búsqueda), lectura de .txt y archivos de salida."""

import os
import re
import shutil
import time
import unicodedata
from collections import deque
from pathlib import Path

if "ErrorUsuario" not in globals():
    raise RuntimeError("Ejecuta primero la celda 1 · Preparar el entorno.")


# ═══ Google Drive ═══════════════════════════════════════════════════════════════

RAIZ_DRIVE = Path("/content/drive")
MI_UNIDAD = RAIZ_DRIVE / "MyDrive"
CARPETA_TRABAJO = Path("/content/clonador_voz")
EXTENSIONES_MEDIOS = frozenset({
    ".wav", ".mp3", ".m4a", ".aac", ".ogg", ".oga", ".opus", ".flac", ".wma", ".amr",
    ".webm", ".mp4", ".mov", ".mkv", ".avi", ".3gp",
})
EXTENSIONES_TEXTO = frozenset({".txt", ".md"})
CARPETA_PROYECTO = "Clonador de voz"  # carpeta de Mi unidad para audios, voces guardadas y modelos entrenados
PREFIJOS_MI_UNIDAD = ("/content/drive/MyDrive/", "MyDrive/", "My Drive/", "Mi unidad/")
NOMBRES_MI_UNIDAD = frozenset({"/content/drive/MyDrive", "MyDrive", "My Drive", "Mi unidad"})


INSTRUCCIONES_DRIVE = (
    "No se pudo conectar tu Google Drive. Vuelve a ejecutar la celda y, en la ventana de Google: "
    "1) elige la MISMA cuenta con la que abriste Colab y 2) marca TODOS los permisos («Seleccionar todo») antes de "
    "«Continuar». Si sigue fallando: menú «Entorno de ejecución → Desconectar y eliminar entorno de ejecución» "
    "(Runtime → Disconnect and delete runtime) y empieza desde la celda 1, o conecta Drive con el botón de Drive del "
    "panel «Archivos» (ícono de carpeta, a la izquierda). En la celda 4 también puedes subir el audio desde tu computadora."
)


class DriveNoDisponible(ErrorUsuario):
    """Drive no se pudo montar: quien llama decide si es un error (leer un archivo) o solo un aviso (guardar)."""

    def __init__(self, detalle: str) -> None:
        super().__init__(f"{INSTRUCCIONES_DRIVE} (Detalle técnico: {detalle})")
        self.detalle = detalle


def drive_montado() -> bool:
    """¿Drive está montado de verdad? Una carpeta «MyDrive» creada a mano no cuenta."""
    return os.path.ismount(RAIZ_DRIVE) and MI_UNIDAD.is_dir()


def montar_drive(intentos: int = 2) -> Path:
    """Monta Google Drive (la primera vez pide permiso) y devuelve «Mi unidad».

    «mount failed» suele deberse a permisos incompletos, a otra cuenta o a un fallo pasajero: se reintenta una vez
    y, si persiste, se explica qué hacer.
    """
    if drive_montado():
        return MI_UNIDAD
    from google.colab import drive

    ultimo_error: Exception | None = None
    for intento in range(1, intentos + 1):
        try:
            drive.mount(str(RAIZ_DRIVE), force_remount=intento > 1)
        except Exception as error:  # noqa: BLE001 — ValueError («mount failed»), MessageError («credential propagation…»)
            ultimo_error = error
            if intento < intentos:
                print(f"⚠️ Drive no se conectó ({error}). Reintentando; si aparece la ventana de Google, marca todos los permisos.")
            continue
        if drive_montado():
            return MI_UNIDAD
        ultimo_error = RuntimeError("Drive respondió, pero «Mi unidad» no apareció")
    raise DriveNoDisponible(f"{type(ultimo_error).__name__}: {ultimo_error}")


def clave_nombre(nombre: str) -> str:
    """Forma comparable de un nombre: Unicode NFC y sin mayúsculas.

    Drive conserva tal cual los nombres de archivos subidos desde Mac, cuyas tildes vienen en forma NFD.
    """
    return unicodedata.normalize("NFC", nombre).casefold()


def ruta_en_mi_unidad(texto: str) -> Path:
    """Interpreta lo escrito como ruta dentro de «Mi unidad»; solo una ruta completa de /content sale de ella."""
    limpio = unicodedata.normalize("NFC", texto.strip().strip("\"'").strip())
    if limpio.rstrip("/") in NOMBRES_MI_UNIDAD:
        return MI_UNIDAD
    for prefijo in PREFIJOS_MI_UNIDAD:
        if limpio.startswith(prefijo):
            return MI_UNIDAD / limpio[len(prefijo):]
    if limpio.startswith("/content/"):
        return Path(limpio)  # p. ej. una unidad compartida: /content/drive/Shareddrives/…
    return MI_UNIDAD / limpio.lstrip("/")  # «/Carpeta» también es relativa a Mi unidad


def carpeta_del_proyecto(*partes: str) -> Path:
    """«Mi unidad/Clonador de voz/…» (monta Drive si hace falta y crea la carpeta)."""
    montar_drive()
    return preparar_carpeta(MI_UNIDAD.joinpath(CARPETA_PROYECTO, *partes))


def describir_ruta(ruta: Path) -> str:
    """Ruta legible: «Mi unidad/…» para Drive y la ruta completa en otro caso."""
    try:
        return f"Mi unidad/{ruta.relative_to(MI_UNIDAD)}"
    except ValueError:
        return str(ruta)


def resolver_en_drive(entrada: str, *, extensiones: frozenset[str], incluir_carpetas: bool) -> Path:
    """La ruta indicada (tolera mayúsculas, formas Unicode y extensión omitida) o, si es solo un nombre, lo busca."""
    limpio = entrada.strip().strip("\"'").strip()
    if not limpio:
        raise ErrorUsuario("Falta el nombre del archivo en Drive.")
    ruta = ruta_en_mi_unidad(limpio)
    if ruta.exists():
        return ruta
    if "/" not in limpio:
        return buscar_por_nombre(limpio, MI_UNIDAD, extensiones=extensiones, incluir_carpetas=incluir_carpetas)
    encontrada = resolver_por_partes(ruta, extensiones)
    if encontrada is None:
        raise ErrorUsuario(f"No existe «{limpio}» en Mi unidad. Revisa la ruta y el nombre del archivo.")
    print(f"📄 Usando {describir_ruta(encontrada)}")
    return encontrada


def resolver_por_partes(ruta: Path, extensiones: frozenset[str]) -> Path | None:
    """Recorre la ruta tramo a tramo: el tramo que no existe tal cual se busca entre sus hermanos por su clave."""
    actual = Path(ruta.anchor)
    partes = ruta.parts[1:]
    for indice, parte in enumerate(partes):
        exacta = actual / parte
        if exacta.exists():
            actual = exacta
            continue
        objetivo, ultimo = clave_nombre(parte), indice == len(partes) - 1
        try:
            hermanos = sorted(os.scandir(actual), key=lambda entrada: entrada.name)
        except OSError:
            return None
        coincidencia = next(
            (
                h for h in hermanos
                if clave_nombre(h.name) == objetivo or (ultimo and h.is_file() and _coincide(h.name, objetivo, extensiones))
            ),
            None,
        )
        if coincidencia is None:
            return None
        actual = Path(coincidencia.path)
    return actual


def _coincide(nombre: str, objetivo: str, extensiones: frozenset[str]) -> bool:
    """Igualdad por clave de nombre; si se omitió la extensión, acepta cualquiera de las válidas."""
    nombre = clave_nombre(nombre)
    if nombre == objetivo:
        return True
    raiz, extension = os.path.splitext(nombre)
    return raiz == objetivo and extension in extensiones


def _fecha_modificacion(ruta: Path) -> float:
    try:
        return ruta.stat().st_mtime
    except OSError:
        return 0.0


def buscar_por_nombre(
    nombre: str,
    raiz: Path,
    *,
    extensiones: frozenset[str],
    incluir_carpetas: bool,
    limite_s: float = 60.0,
    margen_s: float = 4.0,
) -> Path:
    """Búsqueda en anchura (primero lo menos profundo) con tiempo límite; ante duplicados, el más reciente."""
    objetivo = clave_nombre(nombre)
    print(f"🔎 Buscando «{nombre}» en tu Drive…")
    encontrados: list[Path] = []
    pendientes: deque[Path] = deque([raiz])
    fin = time.monotonic() + limite_s
    while pendientes and time.monotonic() < fin:
        try:
            entradas = list(os.scandir(pendientes.popleft()))
        except OSError:
            continue
        for entrada in entradas:
            if entrada.name.startswith("."):
                continue
            if entrada.is_dir(follow_symlinks=False):
                pendientes.append(Path(entrada.path))
                if incluir_carpetas and clave_nombre(entrada.name) == objetivo:
                    encontrados.append(Path(entrada.path))
            elif _coincide(entrada.name, objetivo, extensiones):
                encontrados.append(Path(entrada.path))
        if encontrados:  # margen breve para detectar homónimos y terminar
            fin = min(fin, time.monotonic() + margen_s)
    if not encontrados:
        raise ErrorUsuario(f"No encontré «{nombre}» en Mi unidad. Escribe su ruta, p. ej. «Carpeta/{nombre}».")
    elegido = max(encontrados, key=_fecha_modificacion)
    if len(encontrados) > 1:
        print(f"ℹ️ Hay {len(encontrados)} coincidencias; uso la más reciente. Para usar otra, escribe su ruta:")
        for ruta in encontrados[:5]:
            print("   •", describir_ruta(ruta))
    print(f"📄 Usando {describir_ruta(elegido)}")
    return elegido


def listar_medios(ruta: Path, maximo: int = 12, *, subcarpetas: bool = False) -> list[Path]:
    """El archivo indicado, o los audios y videos de una carpeta en orden alfabético (con sus subcarpetas, si se pide)."""
    if ruta.is_file():
        return [ruta]
    candidatos = ruta.rglob("*") if subcarpetas else ruta.iterdir()
    archivos = sorted(
        (p for p in candidatos if p.is_file() and p.suffix.casefold() in EXTENSIONES_MEDIOS and not p.name.startswith(".")),
        key=lambda p: str(p.relative_to(ruta)).casefold(),
    )
    if not archivos:
        raise ErrorUsuario(f"La carpeta «{ruta.name}» no tiene audios ni videos.")
    if len(archivos) > maximo:
        print(f"ℹ️ La carpeta tiene {len(archivos)} archivos; uso los primeros {maximo}.")
    return archivos[:maximo]


def leer_texto_de_archivo(ruta: Path) -> str:
    """UTF-8 (con o sin BOM); si no lo es, Windows-1252 (Bloc de notas antiguo) y, como último recurso, Latin-1."""
    datos = ruta.read_bytes()
    for codificacion in ("utf-8-sig", "cp1252"):
        try:
            return datos.decode(codificacion)
        except UnicodeDecodeError:
            continue
    return datos.decode("latin-1")


# ═══ Archivos de salida ═════════════════════════════════════════════════════════


def preparar_carpeta(ruta: Path, *, vaciar: bool = False) -> Path:
    """Crea la carpeta (y sus padres); con vaciar=True la deja vacía antes."""
    if vaciar and ruta.exists():
        shutil.rmtree(ruta)
    ruta.mkdir(parents=True, exist_ok=True)
    return ruta


def nombre_seguro(nombre: str, por_defecto: str = "voz_clonada") -> str:
    """Nombre de archivo válido: sin extensión de audio o subtítulos ni caracteres prohibidos."""
    base = re.sub(r"\.(wav|mp3|ogg|opus|srt)$", "", nombre.strip(), flags=re.IGNORECASE)
    base = re.sub(r'[\\/:*?"<>|\x00-\x1f]+', "_", base).strip(" ._")
    return base[:120] or por_defecto


def base_libre(carpeta: Path, base: str, extensiones: tuple[str, ...]) -> str:
    """Primer nombre (nombre, nombre_2, nombre_3…) libre con TODAS las extensiones: el audio y sus subtítulos
    quedan con el mismo nombre y nada se sobrescribe."""
    candidato, n = base, 2
    while any((carpeta / f"{candidato}{extension}").exists() for extension in extensiones):
        candidato, n = f"{base}_{n}", n + 1
    return candidato


def ruta_libre(carpeta: Path, base: str, extension: str) -> Path:
    """Primera ruta que no sobrescribe nada: nombre.ext, nombre_2.ext, nombre_3.ext…"""
    return carpeta / f"{base_libre(carpeta, base, (extension,))}{extension}"


def formatear_duracion(segundos: float) -> str:
    minutos, resto = divmod(segundos, 60.0)
    return f"{int(minutos)} min {resto:.0f} s" if minutos >= 1 else f"{resto:.1f} s"


# ─── Ejecución ───
print("✅ Funciones de archivos cargadas.")

In [ ]:
# @title 2b · Funciones: audio y volumen { display-mode: "form" }
# @markdown Conversión con FFmpeg, procesamiento del audio con NumPy, volumen parejo (LUFS) y música de fondo. No tiene parámetros: solo ejecútala.

"""Celda 2b · Audio: FFmpeg, WAV y procesamiento con NumPy, sonoridad ITU-R BS.1770-4 con limitador, remuestreo,
control de calidad (silencios), mezcla con música de fondo que se atenúa bajo la voz y reproductor."""

import subprocess
import wave
from collections.abc import Sequence
from pathlib import Path
from typing import Any

import numpy as np

if "preparar_carpeta" not in globals():
    raise RuntimeError("Ejecuta primero las celdas 1 y 2a.")


# ═══ Audio con FFmpeg ═══════════════════════════════════════════════════════════

FRECUENCIA_REFERENCIA = 22050  # Hz con los que XTTS analiza la voz de referencia


def filtros_referencia(limpiar: bool, reducir_ruido: bool) -> str:
    """Cadena de filtros de FFmpeg para preparar la voz de referencia."""
    filtros: list[str] = []
    if limpiar:
        filtros.append("highpass=f=70")  # retumbes y ruido grave
    if reducir_ruido:
        filtros.append("afftdn=nr=10:nf=-30")  # reducción espectral moderada
    if limpiar:
        recorte = "silenceremove=start_periods=1:start_duration=0.05:start_threshold=-45dB"
        filtros += [recorte, "areverse", recorte, "areverse"]  # silencio inicial y final
        filtros.append("loudnorm=I=-20:TP=-2:LRA=11")  # volumen homogéneo (EBU R128)
    return ",".join(filtros)


def ejecutar_ffmpeg(argumentos: Sequence[str], tarea: str) -> None:
    """Ejecuta FFmpeg sin ruido en pantalla; si falla, explica qué no pudo hacer."""
    comando = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-nostdin", "-y", *argumentos]
    resultado = subprocess.run(comando, capture_output=True, text=True, check=False)
    if resultado.returncode != 0:
        detalle = (resultado.stderr.strip().splitlines() or ["sin detalle"])[-1]
        raise ErrorUsuario(f"FFmpeg no pudo {tarea}: {detalle}")


def preparar_referencia(
    origen: Path, destino: Path, *, filtros: str, max_segundos: float, frecuencia: int = FRECUENCIA_REFERENCIA
) -> None:
    """Cualquier audio o video → WAV mono de 16 bits a la frecuencia indicada, filtrado y recortado."""
    argumentos = ["-t", f"{max_segundos + 30:.0f}", "-i", str(origen), "-vn", "-ac", "1"]
    if filtros:
        argumentos += ["-af", filtros]
    argumentos += ["-ar", str(frecuencia), "-t", f"{max_segundos:.2f}", "-c:a", "pcm_s16le", str(destino)]
    ejecutar_ffmpeg(argumentos, f"leer «{origen.name}» como audio")


def convertir_a_mp3(origen: Path, destino: Path, kbps: int = 192) -> None:
    """WAV → MP3 a 44,1 kHz: a los 24 kHz del modelo, el MP3 no admite 192 kbps y bajaría a 160 sin avisar."""
    argumentos = ["-i", str(origen), "-ar", "44100", "-c:a", "libmp3lame", "-b:a", f"{kbps}k", str(destino)]
    ejecutar_ffmpeg(argumentos, "crear el MP3")


def convertir_a_ogg(origen: Path, destino: Path, kbps: int = 96) -> None:
    """WAV → Ogg Opus (el formato de las notas de voz): 96 kbps a 48 kHz, la frecuencia nativa de Opus."""
    argumentos = ["-i", str(origen), "-ar", "48000", "-c:a", "libopus", "-b:a", f"{kbps}k", str(destino)]
    ejecutar_ffmpeg(argumentos, "crear el OGG")


def decodificar_audio(ruta: Path, frecuencia: int, canales: int = 1, max_segundos: float | None = None) -> np.ndarray:
    """Cualquier audio o video → float32 a la frecuencia pedida, por una tubería de FFmpeg (sin archivos temporales).

    Devuelve (n,) en mono o (n, canales).
    """
    limite = ["-t", f"{max_segundos:.2f}"] if max_segundos else []
    comando = [
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-nostdin", *limite, "-i", str(ruta), "-vn",
        "-ac", str(canales), "-ar", str(frecuencia), "-f", "f32le", "-",
    ]
    resultado = subprocess.run(comando, capture_output=True, check=False)
    if resultado.returncode != 0 or not resultado.stdout:
        detalle = (resultado.stderr.decode(errors="replace").strip().splitlines() or ["sin audio"])[-1]
        raise ErrorUsuario(f"FFmpeg no pudo leer «{ruta.name}» como audio: {detalle}")
    muestras = np.frombuffer(resultado.stdout, dtype="<f4").astype(np.float32)
    return muestras if canales == 1 else muestras.reshape(-1, canales)


def procesar_con_ffmpeg(x: np.ndarray, frecuencia: int, filtro: str) -> np.ndarray:
    """Aplica un filtro de FFmpeg (p. ej. «atempo=1.2») a una señal mono en memoria."""
    formato = ["-f", "f32le", "-ac", "1", "-ar", str(frecuencia)]
    comando = ["ffmpeg", "-hide_banner", "-loglevel", "error", *formato, "-i", "-", "-af", filtro, *formato, "-"]
    resultado = subprocess.run(
        comando, input=np.ascontiguousarray(x, dtype="<f4").tobytes(), capture_output=True, check=False
    )
    if resultado.returncode != 0:
        detalle = (resultado.stderr.decode(errors="replace").strip().splitlines() or ["sin detalle"])[-1]
        raise ErrorUsuario(f"FFmpeg no pudo aplicar «{filtro}»: {detalle}")
    return np.frombuffer(resultado.stdout, dtype="<f4").astype(np.float32)


# ═══ WAV y procesamiento con NumPy ══════════════════════════════════════════════


def leer_wav(ruta: Path) -> tuple[np.ndarray, int]:
    """WAV PCM de 16 bits → (muestras float32 en [-1, 1], frecuencia). Sin librerías de audio."""
    with wave.open(str(ruta), "rb") as wav:
        canales, ancho, frecuencia = wav.getnchannels(), wav.getsampwidth(), wav.getframerate()
        datos = wav.readframes(wav.getnframes())
    if ancho != 2:
        raise ValueError(f"{ruta.name}: se esperaba PCM de 16 bits")
    muestras = np.frombuffer(datos, dtype="<i2").astype(np.float32) / np.float32(32768.0)
    if canales > 1:
        muestras = muestras.reshape(-1, canales).mean(axis=1, dtype=np.float32)
    return muestras, frecuencia


def escribir_wav(ruta: Path, muestras: np.ndarray, frecuencia: int) -> None:
    """float32 en [-1, 1] → WAV PCM de 16 bits: (n,) mono o (n, canales) intercalado."""
    canales = 1 if muestras.ndim == 1 else muestras.shape[1]
    pcm = np.ascontiguousarray(np.round(np.clip(muestras, -1.0, 1.0) * 32767.0).astype("<i2"))
    with wave.open(str(ruta), "wb") as wav:
        wav.setnchannels(canales)
        wav.setsampwidth(2)
        wav.setframerate(frecuencia)
        wav.writeframes(pcm.tobytes())


def rms_por_tramas(x: np.ndarray, ventana: int, salto: int) -> np.ndarray:
    """RMS de tramas solapadas en O(n) con una suma acumulada de cuadrados (sin copiar tramas)."""
    if x.size == 0:
        return np.zeros(0)
    if x.size <= ventana:
        return np.sqrt([np.mean(np.square(x, dtype=np.float64))])
    acumulada = np.concatenate(([0.0], np.cumsum(np.square(x, dtype=np.float64))))
    inicios = np.arange(0, x.size - ventana + 1, salto)
    energia = (acumulada[inicios + ventana] - acumulada[inicios]) / ventana
    return np.sqrt(np.maximum(energia, 0.0))


def recortar_silencios(x: np.ndarray, frecuencia: int, umbral_db: float = -45.0, margen_ms: float = 60.0) -> np.ndarray:
    """Quita el silencio inicial y final: conserva de la primera a la última trama con voz."""
    ventana, salto = max(1, frecuencia // 50), max(1, frecuencia // 100)  # tramas de 20 ms cada 10 ms
    rms = rms_por_tramas(x, ventana, salto)
    if rms.size == 0 or rms.max() < 1e-5:
        return x[:0]
    activas = np.flatnonzero(rms >= rms.max() * 10.0 ** (umbral_db / 20.0))
    margen = int(frecuencia * margen_ms / 1000.0)
    inicio = max(0, int(activas[0]) * salto - margen)
    fin = min(x.size, int(activas[-1]) * salto + ventana + margen)
    return x[inicio:fin]


def aplicar_fundidos(x: np.ndarray, frecuencia: int, ms: float = 8.0) -> np.ndarray:
    """Fundido de entrada y de salida muy corto para que las uniones no hagan «clic»."""
    n = min(int(frecuencia * ms / 1000.0), x.size // 2)
    if n == 0:
        return x
    y = x.astype(np.float32, copy=True)
    rampa = np.linspace(0.0, 1.0, n, dtype=np.float32)
    y[:n] *= rampa
    y[-n:] *= rampa[::-1]
    return y


def normalizar_pico(x: np.ndarray, dbfs: float = -1.0) -> np.ndarray:
    """Escala la señal para que su pico quede en `dbfs` (−1 dBFS deja margen al MP3)."""
    pico = float(np.max(np.abs(x))) if x.size else 0.0
    if pico < 1e-6:
        return x.astype(np.float32, copy=False)
    return (x * np.float32(10.0 ** (dbfs / 20.0) / pico)).astype(np.float32, copy=False)


def ensamblar(
    fragmentos: Sequence[np.ndarray], pausas_s: Sequence[float], frecuencia: int
) -> tuple[np.ndarray, list[tuple[int, int]]]:
    """Une los fragmentos dejando tras cada uno su pausa (reserva la memoria una sola vez) y devuelve también
    dónde quedó cada uno, en muestras (inicio, fin): con eso se sincronizan los subtítulos."""
    pausas = [max(0, round(p * frecuencia)) for p in pausas_s]
    salida = np.zeros(sum(f.size for f in fragmentos) + sum(pausas), dtype=np.float32)
    posiciones: list[tuple[int, int]] = []
    posicion = 0
    for fragmento, pausa in zip(fragmentos, pausas, strict=True):
        salida[posicion : posicion + fragmento.size] = fragmento
        posiciones.append((posicion, posicion + fragmento.size))
        posicion += fragmento.size + pausa
    return salida, posiciones


def concatenar_con_pausas(fragmentos: Sequence[np.ndarray], pausas_s: Sequence[float], frecuencia: int) -> np.ndarray:
    return ensamblar(fragmentos, pausas_s, frecuencia)[0]


def remuestrear(x: np.ndarray, origen: int, destino: int) -> np.ndarray:
    """Cambia la frecuencia de muestreo de una señal mono recortando o ampliando su espectro (filtro ideal).

    El largo de la FFT es múltiplo de origen/mcd(origen, destino), así el largo de salida es exacto; el relleno
    con ceros evita que el final se «envuelva» sobre el principio.
    """
    x = np.asarray(x, dtype=np.float64)
    if origen == destino or x.size == 0:
        return x.astype(np.float32)
    divisor = np.gcd(origen, destino)
    paso = origen // divisor
    largo = -(-(x.size + origen // 50) // paso) * paso  # ≥ 20 ms de relleno
    largo_salida = largo * destino // origen
    espectro = np.fft.rfft(x, largo)
    nuevo = np.zeros(largo_salida // 2 + 1, dtype=np.complex128)
    comunes = min(espectro.size, nuevo.size)
    nuevo[:comunes] = espectro[:comunes]
    if largo_salida < largo and largo_salida % 2 == 0:
        nuevo[-1] = 0.0  # al bajar la frecuencia, el bin de Nyquist no tiene un equivalente exacto
    y = np.fft.irfft(nuevo, largo_salida) * (largo_salida / largo)
    return y[: round(x.size * destino / origen)].astype(np.float32)


def silencio_interno_maximo(x: np.ndarray, frecuencia: int, umbral_db: float = -40.0) -> float:
    """Segundos del silencio más largo ENTRE dos tramos con voz (los bordes no cuentan): delata los fragmentos en
    que el modelo se «queda callado» a mitad de frase."""
    ventana, salto = max(1, frecuencia // 50), max(1, frecuencia // 100)
    rms = rms_por_tramas(np.asarray(x, dtype=np.float32), ventana, salto)
    if rms.size == 0 or rms.max() < 1e-5:
        return 0.0
    activas = np.flatnonzero(rms >= rms.max() * 10.0 ** (umbral_db / 20.0))
    if activas.size < 2:
        return 0.0
    return float((np.diff(activas) - 1).max()) * salto / frecuencia


# ═══ Volumen: sonoridad ITU-R BS.1770-4 (EBU R128) y limitador ══════════════════

GANANCIA_MAXIMA_DB = 30.0  # no se amplifica más que esto aunque el audio venga muy bajo


def coeficientes_filtro_k(frecuencia: int) -> tuple[tuple[np.ndarray, np.ndarray], tuple[np.ndarray, np.ndarray]]:
    """(b, a) de las dos etapas del filtro K (estante de agudos + pasa altos) para cualquier frecuencia de muestreo.

    Fórmulas de libebur128; a 48 kHz reproducen los coeficientes tabulados en la norma.
    """
    f0, ganancia_db, q = 1681.974450955533, 3.999843853973347, 0.7071752369554196
    k = np.tan(np.pi * f0 / frecuencia)
    vh = 10.0 ** (ganancia_db / 20.0)
    vb = vh**0.4996667741545416
    a0 = 1.0 + k / q + k * k
    estante = (
        np.array([(vh + vb * k / q + k * k) / a0, 2.0 * (k * k - vh) / a0, (vh - vb * k / q + k * k) / a0]),
        np.array([1.0, 2.0 * (k * k - 1.0) / a0, (1.0 - k / q + k * k) / a0]),
    )
    f0, q = 38.13547087602444, 0.5003270373238773
    k = np.tan(np.pi * f0 / frecuencia)
    a0 = 1.0 + k / q + k * k
    pasa_altos = (np.array([1.0, -2.0, 1.0]), np.array([1.0, 2.0 * (k * k - 1.0) / a0, (1.0 - k / q + k * k) / a0]))
    return estante, pasa_altos


def largo_fft_rapido(n: int) -> int:
    """Menor 2^a·3^b·5^c ≥ n: con esos tamaños la FFT es rápida y casi no se rellena de más."""
    mejor = 1 << max(0, n - 1).bit_length()
    potencia5 = 1
    while potencia5 < mejor:
        potencia35 = potencia5
        while potencia35 < mejor:
            veces = -(-n // potencia35)  # cuántas veces cabe: se completa con la menor potencia de 2 suficiente
            mejor = min(mejor, potencia35 << (veces - 1).bit_length())
            potencia35 *= 3
        potencia5 *= 5
    return mejor


def ponderar_k(x: np.ndarray, frecuencia: int) -> np.ndarray:
    """Aplica el filtro K multiplicando espectros: con 0,5 s de relleno equivale al filtro IIR, sin bucles en Python.
    Acepta (n,) o (n, canales): cada canal se filtra por separado."""
    largo = largo_fft_rapido(x.shape[0] + frecuencia // 2)
    z = np.exp(-2j * np.pi * np.arange(largo // 2 + 1) / largo)  # z⁻¹ en cada frecuencia de la FFT
    respuesta = np.ones_like(z)
    for b, a in coeficientes_filtro_k(frecuencia):
        respuesta *= (b[0] + z * (b[1] + z * b[2])) / (a[0] + z * (a[1] + z * a[2]))
    if x.ndim == 2:
        respuesta = respuesta[:, None]
    return np.fft.irfft(np.fft.rfft(x, largo, axis=0) * respuesta, largo, axis=0)[: x.shape[0]]


def sonoridad_integrada(x: np.ndarray, frecuencia: int) -> float:
    """Sonoridad integrada (LUFS) de una señal mono o estéreo: bloques de 400 ms con 75 % de solapamiento,
    compuerta absoluta de −70 LUFS y relativa de −10 LU; los canales suman su energía (peso 1, como L y R en la
    norma). Devuelve −inf si todo es silencio."""
    y = ponderar_k(np.asarray(x, dtype=np.float64), frecuencia)
    cuadrados = y * y if y.ndim == 1 else (y * y).sum(axis=1)
    ventana, salto = round(0.4 * frecuencia), round(0.1 * frecuencia)
    acumulada = np.concatenate(([0.0], np.cumsum(cuadrados)))
    if cuadrados.size >= ventana:
        inicios = np.arange(0, cuadrados.size - ventana + 1, salto)
        energias = (acumulada[inicios + ventana] - acumulada[inicios]) / ventana
    else:
        energias = np.array([acumulada[-1] / max(1, cuadrados.size)])
    energias = np.maximum(energias, 1e-20)
    sonoridades = -0.691 + 10.0 * np.log10(energias)
    energias, sonoridades = energias[sonoridades > -70.0], sonoridades[sonoridades > -70.0]
    if energias.size == 0:
        return float("-inf")
    compuerta = -0.691 + 10.0 * np.log10(energias.mean()) - 10.0
    return float(-0.691 + 10.0 * np.log10(energias[sonoridades > compuerta].mean()))


def minimo_deslizante(x: np.ndarray, ventana: int) -> np.ndarray:
    """m[i] = min(x[i : i + ventana]) en O(n) con el algoritmo de van Herk/Gil-Werman, todo vectorizado:
    cada ventana cae en dos bloques consecutivos, y basta el mínimo-sufijo de uno y el mínimo-prefijo del otro."""
    if ventana <= 1 or x.size == 0:
        return x.astype(np.float64, copy=True)
    bloques = -(-(x.size + ventana - 1) // ventana)
    relleno = np.full(bloques * ventana, np.inf)
    relleno[: x.size] = x
    matriz = relleno.reshape(bloques, ventana)
    prefijo = np.minimum.accumulate(matriz, axis=1).ravel()
    sufijo = np.minimum.accumulate(matriz[:, ::-1], axis=1)[:, ::-1].ravel()
    indices = np.arange(x.size)
    return np.minimum(sufijo[indices], prefijo[indices + ventana - 1])


def limitar_picos(x: np.ndarray, frecuencia: int, techo_dbfs: float = -1.0) -> np.ndarray:
    """Limitador con anticipación: la ganancia baja en rampa antes de cada pico y ninguna muestra pasa el techo.

    g[n] = media de los últimos 7,5 ms de min(ganancia necesaria en los próximos 15 ms): como la media abarca
    menos que el mínimo, en cada pico la ganancia nunca supera la necesaria. En estéreo, una sola ganancia para
    ambos canales (manda el más alto), para que la imagen no se mueva.
    """
    techo = 10.0 ** (techo_dbfs / 20.0)
    x = np.asarray(x)
    amplitud = np.abs(np.asarray(x, dtype=np.float64))
    if amplitud.ndim == 2:
        amplitud = amplitud.max(axis=1)
    if amplitud.size == 0 or amplitud.max() <= techo:
        return np.asarray(x, dtype=np.float32)
    anticipacion, suavizado = max(1, round(0.015 * frecuencia)), max(1, round(0.0075 * frecuencia))
    envolvente = minimo_deslizante(np.minimum(1.0, techo / np.maximum(amplitud, 1e-12)), anticipacion)
    acumulada = np.concatenate(([0.0], np.cumsum(envolvente)))
    fin = np.arange(1, amplitud.size + 1)
    inicio = np.maximum(0, fin - suavizado)
    ganancia = (acumulada[fin] - acumulada[inicio]) / (fin - inicio)
    if x.ndim == 2:
        ganancia = ganancia[:, None]
    return np.clip(x * ganancia, -techo, techo).astype(np.float32)  # el recorte solo absorbe redondeos


def nivelar_volumen(x: np.ndarray, frecuencia: int, objetivo_lufs: float, techo_dbfs: float = -1.0) -> np.ndarray:
    """Lleva la sonoridad integrada al objetivo (volumen percibido parejo) y limita los picos al techo."""
    actual = sonoridad_integrada(x, frecuencia)
    if not np.isfinite(actual):
        return normalizar_pico(x, techo_dbfs)
    ganancia_db = min(objetivo_lufs - actual, GANANCIA_MAXIMA_DB)
    return limitar_picos(np.asarray(x, dtype=np.float64) * 10.0 ** (ganancia_db / 20.0), frecuencia, techo_dbfs)


# ═══ Música de fondo con atenuación automática bajo la voz («ducking») ══════════

ATENUACION_MUSICA_DB = 12.0  # cuánto baja la música mientras hay voz
ENTRADA_MUSICA_S = 1.5  # la música suena sola antes de la primera palabra…
SALIDA_MUSICA_S = 2.5  # …y se desvanece después de la última
TRAMA_S = 0.01


def actividad_de_voz(x: np.ndarray, frecuencia: int, umbral_db: float = -35.0) -> np.ndarray:
    """¿Hay voz en cada trama de 10 ms? (RMS de 20 ms frente al de la trama más fuerte)."""
    ventana, salto = max(1, round(2 * TRAMA_S * frecuencia)), max(1, round(TRAMA_S * frecuencia))
    rms = rms_por_tramas(np.asarray(x, dtype=np.float32), ventana, salto)
    if rms.size == 0 or rms.max() < 1e-5:
        return np.zeros(rms.size, dtype=bool)
    return rms >= rms.max() * 10.0 ** (umbral_db / 20.0)


def dilatar(activas: np.ndarray, antes: int, despues: int) -> np.ndarray:
    """Extiende cada tramo activo `antes` tramas hacia atrás y `despues` hacia adelante (sumas acumuladas, O(n))."""
    acumulada = np.concatenate(([0], np.cumsum(activas, dtype=np.int64)))
    indices = np.arange(activas.size)
    desde = np.clip(indices - despues, 0, activas.size)
    hasta = np.clip(indices + antes + 1, 0, activas.size)
    return acumulada[hasta] - acumulada[desde] > 0


def ganancia_de_atenuacion(activas: np.ndarray, atenuacion_db: float, ataque_s: float = 0.08, relajacion_s: float = 0.6) -> np.ndarray:
    """Ganancia en dB por trama: baja rápido cuando empieza la voz y sube despacio cuando termina (sin «bombeo»)."""
    objetivo = np.where(dilatar(activas, antes=round(0.15 / TRAMA_S), despues=round(0.3 / TRAMA_S)), -atenuacion_db, 0.0)
    ataque, relajacion = np.exp(-TRAMA_S / ataque_s), np.exp(-TRAMA_S / relajacion_s)
    ganancia = np.empty_like(objetivo)
    actual = 0.0
    for i, meta in enumerate(objetivo.tolist()):  # filtro de un polo asimétrico: no se puede vectorizar
        actual = meta + (ataque if meta < actual else relajacion) * (actual - meta)
        ganancia[i] = actual
    return ganancia


def mezclar_con_musica(
    voz: np.ndarray,
    musica: np.ndarray,
    frecuencia: int,
    *,
    nivel_db: float,
    atenuacion_db: float = ATENUACION_MUSICA_DB,
    entrada_s: float = ENTRADA_MUSICA_S,
    salida_s: float = SALIDA_MUSICA_S,
) -> np.ndarray:
    """Voz (n,) + música (m,) o (m, canales) → mezcla con la música en bucle, `nivel_db` LU por debajo de la voz
    en las pausas y `atenuacion_db` más abajo mientras se habla; con entrada y salida de música y fundidos.

    Devuelve (entrada + n + salida, canales); la voz va al centro.
    """
    musica = np.asarray(musica, dtype=np.float32)
    canales = 1 if musica.ndim == 1 else musica.shape[1]
    musica = musica.reshape(musica.shape[0], canales)
    if musica.shape[0] == 0:
        raise ErrorUsuario("La música de fondo no tiene audio.")
    entrada, salida = round(entrada_s * frecuencia), round(salida_s * frecuencia)
    total = entrada + voz.size + salida
    vueltas = -(-total // musica.shape[0])
    cama = np.tile(musica, (vueltas, 1))[:total].astype(np.float64)
    pista = np.zeros(total, dtype=np.float64)
    pista[entrada : entrada + voz.size] = voz
    s_voz, s_musica = sonoridad_integrada(voz, frecuencia), sonoridad_integrada(cama, frecuencia)
    if np.isfinite(s_voz) and np.isfinite(s_musica):
        cama *= 10.0 ** ((s_voz + nivel_db - s_musica) / 20.0)
    ganancia_db = ganancia_de_atenuacion(actividad_de_voz(pista, frecuencia), atenuacion_db)
    centros = (np.arange(ganancia_db.size) + 1.0) * TRAMA_S  # centro de cada ventana de 20 ms
    ganancia = 10.0 ** (np.interp(np.arange(total) / frecuencia, centros, ganancia_db) / 20.0)
    fundido = np.ones(total)
    rampa = min(entrada, total // 2) or min(frecuencia, total // 2)
    fundido[:rampa] = np.linspace(0.0, 1.0, rampa)
    caida = min(salida, total // 2)
    if caida:
        fundido[total - caida :] = np.linspace(1.0, 0.0, caida)
    mezcla = cama * (ganancia * fundido)[:, None] + pista[:, None]
    return mezcla.astype(np.float32)


# ═══ Whisper: transcribir lo que se oye ═════════════════════════════════════════

MODELO_WHISPER = "large-v3-turbo"  # rápido, para verificar fragmentos (celda 6)


def obtener_whisper(nombre: str = MODELO_WHISPER) -> Any:
    """faster-whisper en la GPU: se instala y se carga solo la primera vez que hace falta."""
    modelos = globals().setdefault("_WHISPER", {})
    if nombre not in modelos:
        instalar_si_falta("faster-whisper", f"faster-whisper=={VERSION_FASTER_WHISPER}")
        from faster_whisper import WhisperModel

        print(f"⏳ Cargando Whisper ({nombre}) en la GPU (la primera vez lo descarga)…")
        modelos[nombre] = WhisperModel(nombre, device="cuda", compute_type="float16")
    return modelos[nombre]


def liberar_whisper() -> None:
    """Libera la memoria de GPU de los modelos de Whisper cargados."""
    globals().get("_WHISPER", {}).clear()
    import gc

    gc.collect()
    try:
        import torch
    except ImportError:
        return
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def transcribir(onda: np.ndarray, frecuencia: int, idioma: str) -> str:
    """Lo que Whisper entiende de un fragmento (decodificación voraz: rápida y determinista)."""
    audio = remuestrear(onda, frecuencia, 16000)
    trozos, _ = obtener_whisper().transcribe(
        audio, language=idioma.split("-")[0], beam_size=1, condition_on_previous_text=False, without_timestamps=True
    )
    return " ".join(trozo.text.strip() for trozo in trozos)


LIMITE_REPRODUCTOR_MB = 8.0  # el reproductor incrusta el audio en el cuaderno: más grande lo vuelve pesado
SEGUNDOS_VISTA_PREVIA = 60


def mostrar_reproductor(ruta: Path) -> None:
    """Reproductor dentro de la celda; si el archivo es grande, solo suena su primer minuto."""
    from IPython.display import Audio, display

    if ruta.stat().st_size > LIMITE_REPRODUCTOR_MB * 1_000_000:
        vista = preparar_carpeta(CARPETA_TRABAJO / "vistas_previas") / f"{ruta.stem}.mp3"
        argumentos = ["-t", str(SEGUNDOS_VISTA_PREVIA), "-i", str(ruta), "-c:a", "libmp3lame", "-b:a", "96k", str(vista)]
        ejecutar_ffmpeg(argumentos, "crear la vista previa")
        print("ℹ️ Audio largo: aquí suena el primer minuto; el archivo completo está guardado.")
        ruta = vista
    display(Audio(filename=str(ruta)))


# ─── Ejecución ───
print("✅ Funciones de audio cargadas.")

In [ ]:
# @title 2c · Funciones: texto, voces, subtítulos y pronunciación { display-mode: "form" }
# @markdown División en frases y pausas, idiomas y voces mezclados, lotes, subtítulos, montos y cifras en español, reglas de pronunciación y comparación de textos. No tiene parámetros: solo ejecútala.

"""Celda 2c · Texto: frases y pausas, tramos por idioma y por voz, bloques (lotes), subtítulos .srt, montos y cifras
en español, diccionario de pronunciación y tasa de error por caracteres (para verificar lo que se oye)."""

import re
import unicodedata
from collections.abc import Sequence
from dataclasses import dataclass


# ═══ Texto ══════════════════════════════════════════════════════════════════════


@dataclass(frozen=True)
class Segmento:
    """Trozo de texto que el modelo lee de una vez, la pausa (en segundos) que va después, su idioma y su voz.

    `visible` es ese trozo tal como se escribió (para los subtítulos) y `grupo` une los trozos que salieron de una
    misma frase escrita cuando las cifras o las reglas de pronunciación la alargaron más allá del límite del modelo.
    """

    texto: str
    pausa: float
    idioma: str = ""  # vacío: el idioma principal elegido en la celda 5
    voz: str = ""  # vacío: la voz principal (celda 4)
    visible: str = ""
    grupo: int = 0


PATRON_PAUSA = re.compile(r"\[\s*pausa(?:\s*[=:]?\s*(\d+(?:[.,]\d+)?)\s*s?)?\s*\]", re.IGNORECASE)
PATRON_FIN_FRASE = re.compile(r"(?<=[.!?…।؟])\s+")
PATRON_CORTE_SUAVE = re.compile(r"(?<=[,;:—–])\s+")
PAUSA_MARCA = 1.0  # segundos de un [pausa] sin número
MINIMO_CARACTERES = 20  # frases más cortas se unen a la siguiente: XTTS falla con textos muy breves
# Un punto tras estas palabras (o tras una sola letra: «p. ej.», «J. K. Rowling») no cierra la frase.
ABREVIATURAS = frozenset({
    "sr", "sra", "srta", "sres", "dr", "dra", "dres", "ing", "lic", "mg", "mag", "mtro", "mtra", "prof", "profa",
    "arq", "abg", "abog", "gral", "cnel", "sto", "sta", "mons", "pdte", "ud", "uds", "mr", "mrs", "ms", "st", "jr",
    "av", "avda", "ca", "cdra", "mz", "lt", "urb", "dpto", "depto", "psje", "nro", "núm", "num", "pág", "págs", "pag",
    "fig", "vol", "ej", "aprox", "máx", "mín", "tel", "cel", "cía", "inc", "vs", "e.g", "i.e", "s/",
})


def pausa_de_parrafo(pausa_frase: float) -> float:
    return max(0.6, 2.0 * pausa_frase)


def dividir_texto(texto: str, limite: int, pausa_frase: float) -> list[Segmento]:
    """Texto → segmentos que el modelo puede leer, con la pausa que va tras cada uno.

    [pausa] o [pausa 2.5] fija un silencio; cada salto de línea es un párrafo (pausa más larga).
    """
    pausa_parrafo = pausa_de_parrafo(pausa_frase)
    segmentos: list[Segmento] = []
    for bloque, pausa_marca in _bloques_por_marcas(texto):
        parrafos = [p for p in re.split(r"\n+", bloque) if p.strip()]
        for i, parrafo in enumerate(parrafos):
            frases = _frases(parrafo, limite)
            for j, frase in enumerate(frases):
                if j < len(frases) - 1:
                    pausa = pausa_frase
                else:
                    pausa = pausa_parrafo if i < len(parrafos) - 1 else 0.0
                segmentos.append(Segmento(frase, pausa))
        if segmentos and pausa_marca:
            segmentos[-1] = Segmento(segmentos[-1].texto, segmentos[-1].pausa + pausa_marca)
    return segmentos


def _bloques_por_marcas(texto: str) -> list[tuple[str, float]]:
    """Corta el texto en cada marca de pausa; devuelve (bloque, segundos de la marca que lo sigue)."""
    bloques, inicio = [], 0
    for marca in PATRON_PAUSA.finditer(texto):
        segundos = float(marca.group(1).replace(",", ".")) if marca.group(1) else PAUSA_MARCA
        bloques.append((texto[inicio : marca.start()], segundos))
        inicio = marca.end()
    bloques.append((texto[inicio:], 0.0))
    return bloques


def pausa_inicial(texto: str) -> float:
    """Segundos de las marcas [pausa] que aparecen antes de la primera palabra (todas, si no hay palabras)."""
    total = 0.0
    for bloque, segundos in _bloques_por_marcas(texto):
        if any(caracter.isalnum() for caracter in bloque):
            break
        total += segundos
    return total


def partir_lectura(texto: str, limite: int) -> list[str]:
    """Trozos que el modelo puede leer (sin pausas ni párrafos): sirve para volver a partir una frase que las cifras
    o la pronunciación alargaron."""
    return _frases(texto, limite)


def _frases(parrafo: str, limite: int) -> list[str]:
    """Frases de un párrafo: corta en . ! ? …, parte las demasiado largas y une las muy cortas."""
    limpio = re.sub(r"\s+", " ", parrafo).strip()
    piezas: list[str] = []
    for frase in _dividir_en_frases(limpio):
        piezas.extend(_partir_frase_larga(frase, limite))
    return [frase for frase in _unir_cortas(piezas, limite) if any(c.isalnum() for c in frase)]  # nada de «.» sueltos


def _dividir_en_frases(texto: str) -> list[str]:
    """Corta tras . ! ? … salvo cuando el punto cierra una abreviatura («Dr. Pérez», «Av. Grau», «S/. 150»)."""
    frases: list[str] = []
    inicio = 0
    for corte in PATRON_FIN_FRASE.finditer(texto):
        if _termina_en_abreviatura(texto[inicio : corte.start()]):
            continue
        frases.append(texto[inicio : corte.start()])
        inicio = corte.end()
    frases.append(texto[inicio:])
    return [frase for frase in frases if frase]


def _termina_en_abreviatura(texto: str) -> bool:
    """¿El punto final de `texto` cierra una abreviatura y no una frase?"""
    if not texto.endswith("."):
        return False
    palabra = texto.rsplit(maxsplit=1)[-1].lstrip("([«“\"'¿¡").rstrip(".").casefold()
    return (len(palabra) == 1 and palabra.isalpha()) or palabra in ABREVIATURAS


def _partir_frase_larga(frase: str, limite: int) -> list[str]:
    """Una frase que excede el límite se parte primero en , ; : — y, si no basta, entre palabras."""
    if len(frase) <= limite:
        return [frase]
    trozos: list[str] = []
    for pieza in PATRON_CORTE_SUAVE.split(frase):
        trozos.extend(_partir_por_palabras(pieza, limite) if len(pieza) > limite else [pieza])
    return _empaquetar(trozos, limite)


def _partir_por_palabras(texto: str, limite: int) -> list[str]:
    palabras: list[str] = []
    for palabra in texto.split():
        while len(palabra) > limite:  # una «palabra» enorme (URL, etc.) se corta en seco
            palabras.append(palabra[:limite])
            palabra = palabra[limite:]
        palabras.append(palabra)
    return _empaquetar(palabras, limite)


def _empaquetar(piezas: Sequence[str], limite: int) -> list[str]:
    """Une piezas consecutivas con espacios sin pasar del límite (voraz, respeta el orden)."""
    grupos: list[str] = []
    actual = ""
    for pieza in piezas:
        candidato = f"{actual} {pieza}" if actual else pieza
        if len(candidato) <= limite:
            actual = candidato
        else:
            if actual:
                grupos.append(actual)
            actual = pieza
    if actual:
        grupos.append(actual)
    return grupos


def _unir_cortas(frases: Sequence[str], limite: int) -> list[str]:
    """Une las frases de menos de MINIMO_CARACTERES con la siguiente (o con la anterior si es la última)."""
    resultado: list[str] = []
    pendiente = ""
    for frase in frases:
        if pendiente and len(pendiente) + 1 + len(frase) > limite:
            resultado.append(pendiente)
            pendiente = ""
        actual = f"{pendiente} {frase}" if pendiente else frase
        if len(actual) < MINIMO_CARACTERES:
            pendiente = actual
        else:
            resultado.append(actual)
            pendiente = ""
    if pendiente:
        if resultado and len(resultado[-1]) + 1 + len(pendiente) <= limite:
            resultado[-1] = f"{resultado[-1]} {pendiente}"
        else:
            resultado.append(pendiente)
    return resultado


def partir_por_la_mitad(texto: str) -> tuple[str, str] | None:
    """Corta en el espacio más cercano al centro; sirve para reintentar un fragmento que el modelo rechazó."""
    espacios = [i for i, caracter in enumerate(texto) if caracter == " "]
    if not espacios:
        return None
    corte = min(espacios, key=lambda i: abs(i - len(texto) // 2))
    return texto[:corte].strip(), texto[corte + 1 :].strip()


# ═══ Etiquetas: idiomas y voces ════════════════════════════════════════════════

# Idiomas de los formularios (celda 5 del clonador y celda 3 del entrenamiento). XTTS-v2 también habla chino,
# japonés y coreano, pero requieren pip install "coqui-tts[zh,ja,ko]"; Chatterbox no habla checo ni húngaro.
IDIOMAS = {
    "Español": "es", "English": "en", "Português": "pt", "Français": "fr", "Italiano": "it",
    "Deutsch": "de", "Nederlands": "nl", "Polski": "pl", "Türkçe": "tr", "Русский": "ru",
    "Čeština": "cs", "Magyar": "hu", "العربية": "ar", "हिन्दी": "hi",
}

# [en] … [/en] cambia el idioma; [voz: Nombre] … [/voz] cambia la voz (diálogos). Las de pausa no coinciden.
PATRON_ETIQUETA = re.compile(
    r"\[\s*(?:(?P<cierre>/)?(?P<idioma>[a-z]{2}(?:-[a-z]{2})?)|/\s*voz|voz(?:\s*[:=]\s*(?P<voz>[^\]\n]*))?)\s*\]",
    re.IGNORECASE,
)
PUNTUACION_INICIAL = re.compile(r"^\s*[.,;:!?…»”\"')\]]+")


@dataclass(frozen=True)
class Tramo:
    """Parte continua del texto que se lee en un mismo idioma y con una misma voz."""

    texto: str
    idioma: str
    voz: str = ""  # vacío: la voz principal


def dividir_en_tramos(texto: str, principal: str, disponibles: frozenset[str]) -> tuple[list[Tramo], list[str]]:
    """«Hola [en]Hello![/en] [voz: Ana]¿Qué tal?[/voz]» → tramos (es), (en), (es, Ana).

    `[xx]` pasa al idioma xx y `[/xx]` vuelve al principal; `[voz: Nombre]` cambia de voz y `[/voz]` vuelve a la
    principal. Idioma y voz son independientes; una etiqueta sin cierre vale hasta la siguiente o hasta el final.
    Un código de idioma no disponible se avisa y se lee en el idioma principal.
    """
    tramos: list[Tramo] = []
    avisos: list[str] = []
    idioma, voz, inicio = principal, "", 0
    for etiqueta in PATRON_ETIQUETA.finditer(texto):
        _agregar_tramo(tramos, texto[inicio : etiqueta.start()], idioma, voz)
        inicio = etiqueta.end()
        if etiqueta.group("idioma") is None:
            voz = " ".join((etiqueta.group("voz") or "").split())
            continue
        codigo = etiqueta.group("idioma").lower()
        if etiqueta.group("cierre") or codigo in disponibles:
            idioma = principal if etiqueta.group("cierre") else codigo
        else:
            avisos.append(f"Idioma «{codigo}» no disponible ({etiqueta.group(0)}): esa parte se lee en el idioma principal.")
            idioma = principal
    _agregar_tramo(tramos, texto[inicio:], idioma, voz)
    return tramos, avisos


def _agregar_tramo(tramos: list[Tramo], texto: str, idioma: str, voz: str) -> None:
    """Agrega un tramo. La puntuación que sigue a un cierre («[/en].») y los espacios sueltos van al tramo anterior,
    para no perder la entonación final ni los saltos de párrafo."""
    if tramos:
        anterior = tramos[-1]
        puntuacion = PUNTUACION_INICIAL.match(texto)
        if puntuacion:
            anterior = Tramo(anterior.texto + puntuacion.group(0).strip(), anterior.idioma, anterior.voz)
            texto = texto[puntuacion.end() :]
        if not texto.strip() or (anterior.idioma, anterior.voz) == (idioma, voz):
            tramos[-1] = Tramo(anterior.texto + texto, anterior.idioma, anterior.voz)
            return
        tramos[-1] = anterior
    if texto.strip():
        tramos.append(Tramo(texto, idioma, voz))


def hay_salto_de_parrafo(anterior: str, siguiente: str) -> bool:
    """¿Hay un salto de línea entre el final de un tramo y el inicio del siguiente?"""
    return "\n" in anterior[len(anterior.rstrip()) :] + siguiente[: len(siguiente) - len(siguiente.lstrip())]


# ═══ Lotes: un audio por bloque ═════════════════════════════════════════════════

PATRON_SEPARADOR = re.compile(r"^[ \t]*-{3,}[ \t]*$", re.MULTILINE)  # una línea con solo «---»
PATRON_TITULO = re.compile(r"\A\s*#[ \t]*(?P<titulo>[^\n]*)(?:\n|\Z)")


@dataclass(frozen=True)
class Bloque:
    """Parte del texto que se convierte en un audio aparte; `titulo` (opcional) da nombre al archivo."""

    titulo: str
    texto: str


def dividir_en_bloques(texto: str) -> list[Bloque]:
    """Bloques separados por líneas «---». Si la primera línea de un bloque es «# Título», nombra el archivo y no
    se lee. Se descartan los bloques sin nada pronunciable."""
    bloques: list[Bloque] = []
    for crudo in PATRON_SEPARADOR.split(texto):
        titulo = ""
        encabezado = PATRON_TITULO.match(crudo)
        if encabezado:
            titulo, crudo = encabezado.group("titulo").strip(), crudo[encabezado.end() :]
        if any(caracter.isalnum() for caracter in crudo):
            bloques.append(Bloque(titulo, crudo))
    return bloques


# ═══ Subtítulos .srt ════════════════════════════════════════════════════════════

ANCHO_SUBTITULO = 42  # caracteres por línea (pauta habitual de subtitulado)
LINEAS_SUBTITULO = 2


def _corte_feo(palabra: str) -> bool:
    """Una línea no debe terminar en una palabra de 1–2 caracteres («a», «el», «S/»): queda colgando."""
    return len(palabra) <= 2


def _lineas_voraces(palabras: Sequence[str], ancho: int) -> list[list[str]]:
    """Reparte palabras en líneas de hasta `ancho` caracteres (una palabra más larga ocupa su propia línea); si una
    línea terminaría en palabras cortas («de la»), pasan a la siguiente."""
    lineas: list[list[str]] = []
    largo = 0
    for palabra in palabras:
        if lineas and largo + 1 + len(palabra) <= ancho:
            lineas[-1].append(palabra)
            largo += 1 + len(palabra)
            continue
        arrastre: list[str] = []
        while (
            lineas
            and len(lineas[-1]) > 1
            and _corte_feo(lineas[-1][-1])
            and len(" ".join([lineas[-1][-1], *arrastre, palabra])) <= ancho
        ):
            arrastre.insert(0, lineas[-1].pop())
        lineas.append(arrastre + [palabra])
        largo = len(" ".join(lineas[-1]))
    return lineas


def _en_dos_lineas(palabras: Sequence[str], ancho: int) -> str:
    """Texto de un subtítulo: una línea si cabe; si no, dos lo más parejas posible y sin dejar colgando una palabra
    corta al final de la primera."""
    texto = " ".join(palabras)
    if len(texto) <= ancho or len(palabras) < 2:
        return texto
    largos = [len(" ".join(palabras[:corte])) for corte in range(1, len(palabras))]

    def costo(corte: int) -> int:
        return max(largos[corte - 1], len(texto) - largos[corte - 1] - 1) + (ancho if _corte_feo(palabras[corte - 1]) else 0)

    corte = min(range(1, len(palabras)), key=costo)
    return f"{' '.join(palabras[:corte])}\n{' '.join(palabras[corte:])}"


def _tiempo_srt(segundos: float) -> str:
    milesimas = max(0, round(segundos * 1000))
    horas, milesimas = divmod(milesimas, 3_600_000)
    minutos, milesimas = divmod(milesimas, 60_000)
    segundos_enteros, milesimas = divmod(milesimas, 1000)
    return f"{horas:02d}:{minutos:02d}:{segundos_enteros:02d},{milesimas:03d}"


def subtitulos_srt(entradas: Sequence[tuple[float, float, str]]) -> str:
    """(inicio s, fin s, texto) → archivo .srt. Un texto que no cabe en 2 líneas de 42 caracteres se reparte en
    varios subtítulos, con el tiempo dividido en proporción a sus caracteres."""
    subtitulos: list[tuple[float, float, str]] = []
    for inicio, fin, texto in entradas:
        lineas = _lineas_voraces(texto.split(), ANCHO_SUBTITULO)
        grupos = [sum(lineas[i : i + LINEAS_SUBTITULO], []) for i in range(0, len(lineas), LINEAS_SUBTITULO)]
        total = sum(len(" ".join(grupo)) for grupo in grupos)
        momento = inicio
        for grupo in grupos:
            final = momento + (fin - inicio) * len(" ".join(grupo)) / total
            subtitulos.append((momento, final, _en_dos_lineas(grupo, ANCHO_SUBTITULO)))
            momento = final
    bloques = [
        f"{numero}\n{_tiempo_srt(inicio)} --> {_tiempo_srt(fin)}\n{texto}\n"
        for numero, (inicio, fin, texto) in enumerate(subtitulos, start=1)
    ]
    return "\n".join(bloques)


# ═══ Montos y cifras en español ═════════════════════════════════════════════════
# XTTS no reconoce «S/» y, en español, toma la coma como decimal: lee «1,500» como «uno punto cinco».

_UNIDADES = (
    "cero", "uno", "dos", "tres", "cuatro", "cinco", "seis", "siete", "ocho", "nueve", "diez",
    "once", "doce", "trece", "catorce", "quince", "dieciséis", "diecisiete", "dieciocho", "diecinueve",
    "veinte", "veintiuno", "veintidós", "veintitrés", "veinticuatro", "veinticinco", "veintiséis",
    "veintisiete", "veintiocho", "veintinueve",
)
_DECENAS = ("", "", "", "treinta", "cuarenta", "cincuenta", "sesenta", "setenta", "ochenta", "noventa")
_CENTENAS = (
    "", "ciento", "doscientos", "trescientos", "cuatrocientos", "quinientos",
    "seiscientos", "setecientos", "ochocientos", "novecientos",
)
_ESCALAS = (("", ""), ("millón", "millones"), ("billón", "billones"), ("trillón", "trillones"))


def numero_en_palabras(n: int, *, apocope: bool = False) -> str:
    """Entero ≥ 0 en palabras (escala larga). Con `apocope`, el «uno» final pasa a «un» («veintiún soles»)."""
    if n < 0 or n >= 1000 ** (2 * len(_ESCALAS)):
        raise ValueError(f"Número fuera de rango: {n}")
    if n == 0:
        return "cero"
    grupos: list[str] = []
    for nivel in range(len(_ESCALAS)):
        n, grupo = divmod(n, 1_000_000)
        if grupo:
            grupos.append(_grupo_de_millon(grupo, nivel, apocope and nivel == 0))
        if n == 0:
            break
    return " ".join(reversed(grupos))


def _grupo_de_millon(grupo: int, nivel: int, apocope: bool) -> str:
    """Un grupo de 1 a 999 999 seguido de su escala («millones», «billones»…)."""
    if nivel == 0:
        return _hasta_un_millon(grupo, apocope)
    singular, plural = _ESCALAS[nivel]
    return f"un {singular}" if grupo == 1 else f"{_hasta_un_millon(grupo, True)} {plural}"


def _hasta_un_millon(n: int, apocope: bool) -> str:
    miles, resto = divmod(n, 1000)
    partes: list[str] = []
    if miles == 1:
        partes.append("mil")
    elif miles:
        partes.append(f"{_apocopar(_hasta_mil(miles))} mil")
    if resto:
        palabras = _hasta_mil(resto)
        partes.append(_apocopar(palabras) if apocope else palabras)
    return " ".join(partes)


def _hasta_mil(n: int) -> str:
    """1 a 999."""
    if n == 100:
        return "cien"
    centenas, resto = divmod(n, 100)
    partes = [_CENTENAS[centenas]] if centenas else []
    if resto >= 30:
        decenas, unidades = divmod(resto, 10)
        partes.append(_DECENAS[decenas] + (f" y {_UNIDADES[unidades]}" if unidades else ""))
    elif resto:
        partes.append(_UNIDADES[resto])
    return " ".join(partes)


def _apocopar(palabras: str) -> str:
    """«uno» → «un» y «veintiuno» → «veintiún», como exige el español ante «mil», «millones» o un masculino."""
    if palabras.endswith("veintiuno"):
        return palabras[: -len("veintiuno")] + "veintiún"
    if palabras == "uno" or palabras.endswith(" uno"):
        return palabras[: -len("uno")] + "un"
    return palabras


@dataclass(frozen=True)
class Moneda:
    singular: str
    plural: str
    fraccion_singular: str
    fraccion_plural: str


SOLES = Moneda("sol", "soles", "céntimo", "céntimos")
DOLARES = Moneda("dólar", "dólares", "centavo", "centavos")
_CIFRA = r"\d(?:[\d.,]*\d)?"
_PATRON_SOLES = re.compile(rf"(?<![\w/])S/\.?\s*({_CIFRA})", re.IGNORECASE)
_PATRON_DOLARES = re.compile(rf"(?<!\w)(?:US\s?\$|USD\s?|\$)\s*({_CIFRA})", re.IGNORECASE)
_PATRON_MILES_CON_COMA = re.compile(r"(?<![\d.,])\d{1,3}(?:,\d{3})+(?:\.\d+)?(?![\d,])")


def parsear_monto(cifra: str) -> tuple[int, int] | None:
    """«1,500.50», «1.500,50», «1500» o «12.5» → (entero, céntimos); None si no se entiende.

    Si solo hay un tipo de separador, aparece una vez y no lo siguen exactamente 3 cifras, es decimal;
    en cualquier otro caso es separador de miles («1,500» y «1.500» valen mil quinientos).
    """
    punto, coma = cifra.rfind("."), cifra.rfind(",")
    decimal = ""
    if punto >= 0 and coma >= 0:
        decimal = "." if punto > coma else ","
    elif punto >= 0 or coma >= 0:
        separador = "." if punto >= 0 else ","
        if cifra.count(separador) == 1 and len(cifra.rsplit(separador, 1)[1]) != 3:
            decimal = separador
    entero, _, fraccion = cifra.rpartition(decimal) if decimal else (cifra, "", "")
    entero = entero.replace(".", "").replace(",", "")
    if not entero.isdecimal() or (fraccion and not fraccion.isdecimal()):
        return None
    return int(entero), int((fraccion + "00")[:2]) if fraccion else 0


def monto_en_palabras(entero: int, centimos: int, moneda: Moneda) -> str:
    """(1500, 50, SOLES) → «mil quinientos soles con cincuenta céntimos»; (1, 0, SOLES) → «un sol»."""
    partes: list[str] = []
    if entero or not centimos:
        de = " de" if entero and entero % 1_000_000 == 0 else ""  # «un millón de soles»
        nombre = moneda.singular if entero == 1 else moneda.plural
        partes.append(f"{numero_en_palabras(entero, apocope=True)}{de} {nombre}")
    if centimos:
        nombre = moneda.fraccion_singular if centimos == 1 else moneda.fraccion_plural
        partes.append(f"{numero_en_palabras(centimos, apocope=True)} {nombre}")
    return " con ".join(partes)


def _leer_monto(coincidencia: re.Match[str], moneda: Moneda) -> str:
    monto = parsear_monto(coincidencia.group(1))
    if monto is None:
        return coincidencia.group(0)
    try:
        return monto_en_palabras(*monto, moneda)
    except ValueError:
        return coincidencia.group(0)


def normalizar_cifras(texto: str) -> tuple[str, int]:
    """Montos en soles y dólares → palabras; «1,500» → «1500» para que XTTS lo lea como mil quinientos."""
    total = 0
    for patron, moneda in ((_PATRON_SOLES, SOLES), (_PATRON_DOLARES, DOLARES)):
        texto, cambios = patron.subn(lambda m, moneda=moneda: _leer_monto(m, moneda), texto)
        total += cambios
    texto, cambios = _PATRON_MILES_CON_COMA.subn(lambda m: m.group(0).replace(",", ""), texto)
    return texto, total + cambios


# ═══ Pronunciación ══════════════════════════════════════════════════════════════

LARGO_MAXIMO_REGLA = 100


@dataclass(frozen=True)
class Regla:
    """«texto = lectura»: cada aparición de `texto` (palabra completa, sin distinguir mayúsculas) se lee como `lectura`."""

    texto: str
    lectura: str


def leer_reglas(fuente: str) -> tuple[list[Regla], list[str]]:
    """Reglas separadas por «;» o por líneas; una línea que empieza con «#» es un comentario.

    Si un texto se repite, gana la última regla: así el formulario puede corregir lo que dice el archivo.
    """
    reglas: dict[str, Regla] = {}
    avisos: list[str] = []
    for linea in fuente.splitlines():
        if linea.lstrip().startswith("#"):
            continue
        for crudo in linea.split(";"):
            if not crudo.strip():
                continue
            texto, igual, lectura = (" ".join(parte.split()) for parte in crudo.partition("="))
            if not igual or not texto:
                avisos.append(f"Regla ignorada (formato «texto = lectura»): «{crudo.strip()}»")
                continue
            if len(texto) > LARGO_MAXIMO_REGLA:
                avisos.append(f"Regla ignorada (más de {LARGO_MAXIMO_REGLA} caracteres): «{texto[:40]}…»")
                continue
            texto = unicodedata.normalize("NFC", texto)
            reglas.pop(texto.lower(), None)
            reglas[texto.lower()] = Regla(texto, unicodedata.normalize("NFC", lectura))
    return list(reglas.values()), avisos


class _NodoTrie:
    __slots__ = ("hijos", "fin")

    def __init__(self) -> None:
        self.hijos: dict[str, _NodoTrie] = {}
        self.fin = False


def _patron_de_trie(nodo: _NodoTrie) -> str:
    """Regex equivalente a la trie: prefijos comunes factorizados; el «?» codicioso prueba primero la clave más larga."""
    ramas = [re.escape(caracter) + _patron_de_trie(hijo) for caracter, hijo in sorted(nodo.hijos.items())]
    if not ramas:
        return ""
    cuerpo = ramas[0] if len(ramas) == 1 else "(?:" + "|".join(ramas) + ")"
    return f"(?:{cuerpo})?" if nodo.fin else cuerpo


class Pronunciador:
    """Aplica todas las reglas en una sola pasada; si dos empiezan igual («Machu» / «Machu Picchu»), gana la más larga.

    Los textos se guardan en una trie que se compila a una única regex con los prefijos comunes factorizados:
    el trabajo por posición depende del largo de las claves, no de cuántas reglas haya.
    """

    def __init__(self, reglas: Sequence[Regla]) -> None:
        self.cantidad = len(reglas)
        self._lecturas = {regla.texto.lower(): regla.lectura for regla in reglas}
        raiz = _NodoTrie()
        for clave in self._lecturas:
            nodo = raiz
            for caracter in clave:
                nodo = nodo.hijos.setdefault(caracter, _NodoTrie())
            nodo.fin = True
        self._patron = (
            re.compile(rf"(?<!\w)(?:{_patron_de_trie(raiz)})(?!\w)", re.IGNORECASE) if self._lecturas else None
        )

    def aplicar(self, texto: str) -> tuple[str, int]:
        """Texto con las lecturas aplicadas y cuántos reemplazos hubo."""
        texto = re.sub(r"[ \t]+", " ", unicodedata.normalize("NFC", texto))
        if self._patron is None:
            return texto, 0
        return self._patron.subn(lambda m: self._lecturas.get(m.group(0).lower(), m.group(0)), texto)


# ═══ Comparar lo que se pidió con lo que se oyó (control de calidad con Whisper) ═


def distancia_edicion(a: str, b: str) -> int:
    """Distancia de Levenshtein con el algoritmo de vectores de bits de Myers (versión de Hyyrö): cada columna
    de la matriz de programación dinámica cabe en un entero de Python, así el costo es O(len(b)) operaciones
    sobre enteros de len(a) bits en lugar de O(len(a)·len(b)) pasos en Python."""
    if len(a) < len(b):
        a, b = b, a
    m = len(b)
    if m == 0:
        return len(a)
    mascara, alto = (1 << m) - 1, 1 << (m - 1)
    posiciones: dict[str, int] = {}
    for i, caracter in enumerate(b):
        posiciones[caracter] = posiciones.get(caracter, 0) | (1 << i)
    positivos, negativos, distancia = mascara, 0, m
    for caracter in a:
        iguales = posiciones.get(caracter, 0)
        x_vertical = iguales | negativos
        x_horizontal = (((iguales & positivos) + positivos) ^ positivos) | iguales
        horizontal_positivo = negativos | (~(x_horizontal | positivos) & mascara)
        horizontal_negativo = positivos & x_horizontal
        if horizontal_positivo & alto:
            distancia += 1
        elif horizontal_negativo & alto:
            distancia -= 1
        horizontal_positivo = ((horizontal_positivo << 1) | 1) & mascara  # fila 0: D[0][j] = j
        horizontal_negativo = (horizontal_negativo << 1) & mascara
        positivos = horizontal_negativo | (~(x_vertical | horizontal_positivo) & mascara)
        negativos = horizontal_positivo & x_vertical
    return distancia


def normalizar_para_comparar(texto: str, idioma: str = "") -> str:
    """Minúsculas, sin tildes ni signos y con espacios simples; en español, las cifras pasan a palabras
    (Whisper suele escribir «1500» donde se dijo «mil quinientos»)."""
    if idioma.startswith("es"):
        texto = re.sub(r"\d+", lambda m: numero_en_palabras(int(m.group(0))) if len(m.group(0)) <= 15 else m.group(0), texto)
    sin_tildes = "".join(c for c in unicodedata.normalize("NFKD", texto.casefold()) if not unicodedata.combining(c))
    return " ".join(re.sub(r"[^\w\s]|_", " ", sin_tildes).split())


def tasa_error_caracteres(referencia: str, transcripcion: str, idioma: str = "") -> float:
    """CER: ediciones de caracteres para pasar de lo pedido a lo que se oyó, sobre el largo de lo pedido."""
    pedido = normalizar_para_comparar(referencia, idioma)
    oido = normalizar_para_comparar(transcripcion, idioma)
    return distancia_edicion(pedido, oido) / max(1, len(pedido))


# ─── Ejecución ───
print("✅ Funciones de texto cargadas.")

In [ ]:
# @title 3 · Cargar el modelo de voz en la GPU { display-mode: "form" }
# @markdown **XTTS-v2 · uso asumido por defecto: no comercial** (personal, educativo o de investigación). Es el único que permite su licencia, la Coqui Public Model License ([CPML](https://tts-hub.github.io/cpml)); con la casilla marcada, el cuaderno la acepta en tu nombre.
uso_no_comercial = True  # @param {type:"boolean"}
# @markdown *Desmárcala solo si vas a darle a los audios un uso comercial (anuncios pagados, productos o servicios que vendes): con XTTS-v2 el cuaderno se detendrá antes de cargar el modelo. Para uso comercial elige Chatterbox en la celda 1; con Chatterbox esta casilla no se usa.*
modelo_afinado = ""  # @param {type:"string"}
# @markdown *Opcional, solo XTTS-v2: nombre del modelo que entrenaste con tu voz (cuaderno «Entrenar XTTS con tu voz»), guardado en `Mi unidad/Clonador de voz/modelos`. Vacío = modelo original. El modelo se descarga una vez por sesión.*

"""Celda 3 · Motor de voz: adaptadores con la misma interfaz sobre XTTS-v2 (original o afinado) y Chatterbox.
Todo lo específico de cada modelo vive en esta celda; las demás solo usan MOTOR."""

import gc
import json
import os
import time
from collections.abc import Callable, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np

if not {"montar_drive", "leer_wav", "dividir_texto"} <= globals().keys():  # una función de cada celda 2a, 2b y 2c
    raise RuntimeError("Ejecuta primero las celdas 1, 2a, 2b y 2c.")

NOMBRE_MODELO = "tts_models/multilingual/multi-dataset/xtts_v2"
SEGUNDOS_ESTILO = 30  # audio de referencia que analiza el GPT para imitar entonación y ritmo
TRAMO_ESTILO = 4  # se analiza en tramos de 4 s que luego se promedian: más estable
IDIOMAS_CON_EXTRAS = frozenset({"zh-cn", "ja", "ko"})  # XTTS los habla, pero requieren pip install "coqui-tts[zh,ja,ko]"
LIMITE_CHATTERBOX = 240  # caracteres por fragmento: con más, Chatterbox tiende a saltarse palabras
FORMATO_PERFIL = 1  # versión del formato de los perfiles de voz guardados


@dataclass(frozen=True)
class PerfilVoz:
    """Una voz lista para un modelo concreto.

    `datos`: lo que ese modelo necesita (XTTS: latentes de estilo y embedding; Chatterbox: sus condicionales).
    `huella`: vector normalizado del timbre, para medir el parecido de cada toma (similitud coseno).
    """

    motor: str  # modelo con que se creó: «xtts», «xtts:<modelo afinado>» o «chatterbox»
    datos: Any
    huella: np.ndarray
    archivos: tuple[str, ...]
    segundos: float
    nombre: str = ""


@dataclass(frozen=True)
class AjustesSintesis:
    """Parámetros de una generación; cada motor usa los suyos."""

    idioma: str
    velocidad: float = 1.0
    temperatura: float = 0.75
    top_p: float = 0.85  # XTTS
    top_k: int = 50  # XTTS
    penalizacion_repeticion: float = 5.0  # XTTS
    exageracion: float = 0.5  # Chatterbox: expresividad
    cfg: float = 0.5  # Chatterbox: apego al estilo de la referencia


def a_numpy(tensor: Any) -> np.ndarray:
    return tensor.detach().float().cpu().numpy()


def normalizar_huella(vector: Any) -> np.ndarray:
    v = np.asarray(vector, dtype=np.float64).reshape(-1)
    norma = float(np.linalg.norm(v))
    return (v / norma if norma > 0 else v).astype(np.float32)


def parecido(a: np.ndarray, b: np.ndarray) -> float:
    """Similitud coseno entre dos huellas: 1 = mismo timbre."""
    return float(np.dot(normalizar_huella(a), normalizar_huella(b)))


def metadatos_de_perfil(voz: PerfilVoz) -> dict[str, Any]:
    return {
        "formato": FORMATO_PERFIL,
        "motor": voz.motor,
        "nombre": voz.nombre,
        "segundos": float(voz.segundos),
        "archivos": [Path(archivo).name for archivo in voz.archivos],
        "creado": time.strftime("%Y-%m-%d %H:%M"),
    }


class _MotorBase:
    """Lo común a los dos modelos: semillas, liberar la GPU y validar los WAV de referencia."""

    nombre = ""
    frecuencia_referencia = 0
    extension_perfil = ""

    def __init__(self, modelo: Any, identificador: str) -> None:
        import torch

        self._torch = torch
        self.modelo = modelo
        self.identificador = identificador

    def fijar_semilla(self, semilla: int) -> None:
        """Hace repetible el muestreo aleatorio del modelo."""
        self._torch.manual_seed(semilla)
        if self._torch.cuda.is_available():
            self._torch.cuda.manual_seed_all(semilla)

    def _leer_referencias(self, wavs: Sequence[Path], segundos_por_archivo: float) -> list[np.ndarray]:
        audios = []
        for ruta in wavs:
            muestras, frecuencia = leer_wav(ruta)
            if frecuencia != self.frecuencia_referencia:
                raise ValueError(f"{ruta.name}: {frecuencia} Hz (se esperaban {self.frecuencia_referencia})")
            audios.append(muestras[: int(frecuencia * segundos_por_archivo)])
        return audios


class MotorXTTS(_MotorBase):
    """XTTS-v2 (original o afinado con tu voz) ya cargado en la GPU."""

    nombre = "XTTS-v2"
    frecuencia_referencia = 22050  # Hz con los que XTTS analiza la voz de referencia
    extension_perfil = ".xtts.npz"

    def __init__(self, modelo: Any, identificador: str = "xtts") -> None:
        super().__init__(modelo, identificador)
        self.frecuencia = int(getattr(modelo.args, "output_sample_rate", 24000))
        self.idiomas = frozenset(modelo.config.languages) - IDIOMAS_CON_EXTRAS

    def limite_caracteres(self, idioma: str) -> int:
        """Máximo de caracteres por fragmento que XTTS lee sin cortar el audio."""
        limites = getattr(self.modelo.tokenizer, "char_limits", {})
        return int(limites.get(idioma.split("-")[0], 200))

    def voces_incluidas(self) -> list[str]:
        """Voces de estudio que trae el modelo (para diálogos: [voz: Ana Florence]…[/voz])."""
        gestor = getattr(self.modelo, "speaker_manager", None)
        return sorted(gestor.speakers) if gestor is not None else []

    def voz_incluida(self, nombre: str) -> PerfilVoz:
        latente, embedding = self.modelo.speaker_manager.speakers[nombre].values()  # mismo orden que usa coqui-tts
        dispositivo = self.modelo.device
        datos = (latente.to(dispositivo), embedding.to(dispositivo))
        return PerfilVoz(self.identificador, datos, normalizar_huella(a_numpy(embedding)), (), 0.0, nombre)

    def crear_voz(self, wavs: Sequence[Path], segundos_por_archivo: float, nombre: str = "") -> PerfilVoz:
        """Equivale a Xtts.get_conditioning_latents, pero lee los WAV con NumPy (sin torchaudio.load)."""
        torch, dispositivo = self._torch, self.modelo.device
        audios, embeddings = [], []
        with torch.inference_mode():
            for muestras in self._leer_referencias(wavs, segundos_por_archivo):
                audio = torch.from_numpy(np.ascontiguousarray(muestras)).unsqueeze(0).to(dispositivo)
                embeddings.append(self.modelo.get_speaker_embedding(audio, self.frecuencia_referencia))
                audios.append(audio)
            estilo = self.modelo.get_gpt_cond_latents(
                torch.cat(audios, dim=-1), self.frecuencia_referencia, length=SEGUNDOS_ESTILO, chunk_length=TRAMO_ESTILO
            )
            embedding = torch.stack(embeddings).mean(dim=0)
        total = sum(audio.shape[-1] for audio in audios) / self.frecuencia_referencia
        archivos = tuple(str(ruta) for ruta in wavs)
        return PerfilVoz(self.identificador, (estilo, embedding), normalizar_huella(a_numpy(embedding)), archivos, total, nombre)

    def huella(self, audio: np.ndarray, frecuencia: int) -> np.ndarray:
        """Timbre de un audio generado, con el mismo codificador de hablante que usa el modelo."""
        torch = self._torch
        with torch.inference_mode():
            tensor = torch.from_numpy(np.ascontiguousarray(audio, dtype=np.float32)).unsqueeze(0).to(self.modelo.device)
            return normalizar_huella(a_numpy(self.modelo.get_speaker_embedding(tensor, frecuencia)))

    def guardar_voz(self, voz: PerfilVoz, ruta: Path) -> None:
        latente, embedding = voz.datos
        with open(ruta, "wb") as archivo:
            np.savez(
                archivo,
                latente=a_numpy(latente),
                embedding=a_numpy(embedding),
                huella=voz.huella,
                meta=np.array(json.dumps(metadatos_de_perfil(voz), ensure_ascii=False)),
            )

    def cargar_voz(self, ruta: Path) -> tuple[PerfilVoz, dict[str, Any]]:
        torch, dispositivo = self._torch, self.modelo.device
        with np.load(ruta, allow_pickle=False) as datos:
            meta = json.loads(str(datos["meta"]))
            latente = torch.from_numpy(np.ascontiguousarray(datos["latente"])).to(dispositivo)
            embedding = torch.from_numpy(np.ascontiguousarray(datos["embedding"])).to(dispositivo)
            huella = np.asarray(datos["huella"], dtype=np.float32)
        voz = PerfilVoz(meta["motor"], (latente, embedding), huella, tuple(meta["archivos"]), meta["segundos"], meta["nombre"])
        return voz, meta

    def sintetizar(self, texto: str, voz: PerfilVoz, ajustes: AjustesSintesis) -> np.ndarray:
        """Un fragmento de texto → forma de onda float32 a `self.frecuencia` Hz.

        transformers exige float en temperatura y penalización, y Colab escribe «7» (no «7.0») cuando
        un deslizador cae en un entero: por eso se convierte cada valor a su tipo aquí.
        """
        latente, embedding = voz.datos
        salida = self.modelo.inference(
            texto,
            ajustes.idioma,
            latente,
            embedding,
            temperature=float(ajustes.temperatura),
            top_p=float(ajustes.top_p),
            top_k=int(ajustes.top_k),
            repetition_penalty=float(ajustes.penalizacion_repeticion),
            length_penalty=1.0,
            speed=float(ajustes.velocidad),
            enable_text_splitting=False,  # el texto ya llega dividido por dividir_texto()
        )
        return np.asarray(salida["wav"], dtype=np.float32).reshape(-1)


class MotorChatterbox(_MotorBase):
    """Chatterbox Multilingual (Resemble AI, licencia MIT) ya cargado en la GPU. Sus audios llevan una marca de
    agua inaudible (Perth)."""

    nombre = "Chatterbox"
    frecuencia_referencia = 24000  # la de su decodificador: así la referencia no se remuestrea dos veces
    extension_perfil = ".chatterbox.pt"

    def __init__(self, modelo: Any, identificador: str = "chatterbox") -> None:
        super().__init__(modelo, identificador)
        from chatterbox.mtl_tts import SUPPORTED_LANGUAGES

        self.frecuencia = int(modelo.sr)
        self.idiomas = frozenset(SUPPORTED_LANGUAGES)

    def limite_caracteres(self, idioma: str) -> int:
        return LIMITE_CHATTERBOX

    def voces_incluidas(self) -> list[str]:
        return []  # Chatterbox no trae voces con nombre: en los diálogos se usan tus perfiles guardados

    def voz_incluida(self, nombre: str) -> PerfilVoz:
        raise KeyError(nombre)

    def crear_voz(self, wavs: Sequence[Path], segundos_por_archivo: float, nombre: str = "") -> PerfilVoz:
        """Une las referencias en un solo WAV: Chatterbox toma el timbre de sus primeros 10 s, el estilo de los
        primeros 6 s y el embedding del hablante de todo el audio."""
        audio = np.concatenate(self._leer_referencias(wavs, segundos_por_archivo))
        unida = preparar_carpeta(CARPETA_TRABAJO / "referencias") / "chatterbox_referencia.wav"
        escribir_wav(unida, audio, self.frecuencia_referencia)
        self.modelo.prepare_conditionals(str(unida), exaggeration=0.5)
        total = audio.size / self.frecuencia_referencia
        huella = self.huella(audio, self.frecuencia_referencia)
        return PerfilVoz(self.identificador, self.modelo.conds, huella, tuple(str(r) for r in wavs), total, nombre)

    def huella(self, audio: np.ndarray, frecuencia: int) -> np.ndarray:
        """Embedding del codificador de voz de Chatterbox (trabaja a 16 kHz)."""
        muestras = remuestrear(audio, frecuencia, 16000)
        return normalizar_huella(self.modelo.ve.embeds_from_wavs([muestras], sample_rate=16000, as_spk=True))

    def guardar_voz(self, voz: PerfilVoz, ruta: Path) -> None:
        condicionales = voz.datos
        self._torch.save(
            {
                "t3": dict(condicionales.t3.__dict__),
                "gen": condicionales.gen,
                "huella": self._torch.from_numpy(np.ascontiguousarray(voz.huella)),
                "meta": metadatos_de_perfil(voz),
            },
            ruta,
        )

    def cargar_voz(self, ruta: Path) -> tuple[PerfilVoz, dict[str, Any]]:
        from chatterbox.models.t3.modules.cond_enc import T3Cond
        from chatterbox.mtl_tts import Conditionals

        datos = self._torch.load(ruta, map_location="cpu", weights_only=True)
        meta = datos["meta"]
        condicionales = Conditionals(T3Cond(**datos["t3"]), datos["gen"]).to(self.modelo.device)
        huella = normalizar_huella(a_numpy(datos["huella"]))
        voz = PerfilVoz(meta["motor"], condicionales, huella, tuple(meta["archivos"]), meta["segundos"], meta["nombre"])
        return voz, meta

    def sintetizar(self, texto: str, voz: PerfilVoz, ajustes: AjustesSintesis) -> np.ndarray:
        """Chatterbox no tiene control de velocidad: se aplica después con atempo de FFmpeg (no cambia el tono)."""
        self.modelo.conds = voz.datos
        onda = self.modelo.generate(
            texto,
            language_id=ajustes.idioma,
            exaggeration=float(ajustes.exageracion),
            cfg_weight=float(ajustes.cfg),
            temperature=float(ajustes.temperatura),
        )
        muestras = a_numpy(onda).reshape(-1)
        if abs(float(ajustes.velocidad) - 1.0) > 1e-3:
            muestras = procesar_con_ffmpeg(muestras, self.frecuencia, f"atempo={float(ajustes.velocidad):.4f}")
        return muestras


# ─── Carga de los modelos ───


def liberar_modelo() -> None:
    """Saca el modelo de la GPU (p. ej. antes de entrenar). Después hay que volver a ejecutar las celdas 3 y 4."""
    global _MODELO_CARGADO, MOTOR, VOZ, VOCES
    _MODELO_CARGADO = MOTOR = VOZ = VOCES = None
    gc.collect()
    import torch

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def obtener_modelo(identificador: str, descripcion: str, cargar: Callable[[], Any]) -> Any:
    """Carga el modelo una sola vez por sesión; si se pide otro, libera antes la GPU."""
    global _MODELO_CARGADO
    import torch

    cargado = globals().get("_MODELO_CARGADO")
    if cargado is not None and cargado[0] == identificador:
        print(f"✅ {descripcion} ya estaba cargado en esta sesión.")
        return cargado[1]
    if not torch.cuda.is_available():
        raise ErrorUsuario("PyTorch no detecta la GPU. Cambia el entorno a «T4 GPU» y ejecuta desde la celda 1.")
    liberar_modelo()
    print(f"⏳ Cargando {descripcion} en la GPU…")
    inicio = time.perf_counter()
    modelo = cargar()
    _MODELO_CARGADO = (identificador, modelo)
    print(f"✅ Modelo listo en {torch.cuda.get_device_name(0)} ({time.perf_counter() - inicio:.0f} s)")
    return modelo


def cargar_xtts_original() -> Any:
    from TTS.api import TTS

    print("   La primera vez descarga ~1,9 GB.")
    return TTS(NOMBRE_MODELO, progress_bar=True).to("cuda").synthesizer.tts_model


def carpeta_de_modelo_afinado(nombre: str) -> Path:
    """La carpeta del modelo afinado: un nombre dentro de «Clonador de voz/modelos» o una ruta de Mi unidad."""
    montar_drive()
    limpio = nombre.strip().strip("/")
    for candidata in (MI_UNIDAD / CARPETA_PROYECTO / "modelos" / limpio, ruta_en_mi_unidad(limpio)):
        if all((candidata / archivo).is_file() for archivo in ("model.pth", "config.json", "vocab.json")):
            return candidata
    raise ErrorUsuario(
        f"No encontré el modelo afinado «{limpio}» en Mi unidad/{CARPETA_PROYECTO}/modelos (debe tener model.pth, "
        "config.json y vocab.json). Entrénalo y guárdalo con el cuaderno «Entrenar XTTS con tu voz», o deja vacío "
        "«modelo_afinado»."
    )


def cargar_xtts_afinado(carpeta: Path) -> Any:
    """Como el cuaderno oficial de afinado de coqui-tts: XttsConfig → Xtts → load_checkpoint."""
    from TTS.tts.configs.xtts_config import XttsConfig
    from TTS.tts.models.xtts import Xtts

    configuracion = XttsConfig()
    configuracion.load_json(str(carpeta / "config.json"))
    modelo = Xtts(configuracion)
    modelo.load_checkpoint(configuracion, checkpoint_dir=str(carpeta), eval=True, use_deepspeed=False)
    return modelo.to("cuda")


def cargar_chatterbox() -> Any:
    from chatterbox.mtl_tts import ChatterboxMultilingualTTS

    print("   La primera vez descarga ~3 GB.")
    return ChatterboxMultilingualTTS.from_pretrained(device="cuda")


def preparar_motor(eleccion: str, no_comercial: bool, afinado: str) -> Any:
    """El motor elegido en la celda 1, con la licencia que corresponde."""
    if eleccion == "chatterbox":
        print(
            "ℹ️ Licencia: Chatterbox es MIT (uso comercial permitido). Sus audios llevan una marca de agua inaudible "
            "(Perth) que permite reconocerlos como generados por IA."
        )
        if afinado.strip():
            print("ℹ️ «modelo_afinado» solo aplica a XTTS-v2: se ignora.")
        return MotorChatterbox(obtener_modelo("chatterbox", "Chatterbox Multilingual", cargar_chatterbox))
    confirmar_licencia(no_comercial)
    if afinado.strip():
        carpeta = carpeta_de_modelo_afinado(afinado)
        identificador = f"xtts:{carpeta.name}"
        modelo = obtener_modelo(identificador, f"tu XTTS-v2 afinado «{carpeta.name}»", lambda: cargar_xtts_afinado(carpeta))
        return MotorXTTS(modelo, identificador)
    return MotorXTTS(obtener_modelo("xtts", "XTTS-v2", cargar_xtts_original), "xtts")


# ─── Ejecución ───
MOTOR = None  # sin la licencia aceptada, las celdas 4 a 7 no tienen motor que usar
MOTOR = preparar_motor(MOTOR_ELEGIDO, uso_no_comercial, modelo_afinado)
_incluidas = len(MOTOR.voces_incluidas())
print(
    f"🎛️ {MOTOR.nombre} · {len(MOTOR.idiomas)} idiomas"
    + (f" · {_incluidas} voces incluidas para diálogos" if _incluidas else "")
)

In [ ]:
# @title 4 · Tu voz (grabación, perfil guardado o voz incluida) { display-mode: "form" }
origen_de_la_voz = "Google Drive"  # @param ["Google Drive", "Subir desde mi computadora", "Perfil guardado o voz incluida"]
# @markdown *«Perfil guardado» carga en segundos una voz que ya creaste (sin volver a procesar el audio); también acepta una voz incluida del modelo, p. ej. `Ana Florence`. Si Drive no se conecta, elige «Subir desde mi computadora»: al ejecutar la celda aparecerá un botón para elegir el archivo.*
# @markdown Con Drive, escribe el **nombre del audio** tal como está en tu Drive (`mi_voz.m4a`), una **ruta** dentro de *Mi unidad* (`Audios/mi_voz.m4a`) o una **carpeta** con varios audios tuyos (`Audios/Mi voz`).
audio_en_drive = "mi_voz.wav"  # @param {type:"string"}
# @markdown ---
# @markdown **Perfil de voz**
nombre_del_perfil = "mi_voz"  # @param {type:"string"}
guardar_perfil = True  # @param {type:"boolean"}
# @markdown *Guarda la voz en `Mi unidad/Clonador de voz/voces` con este nombre (reemplaza la anterior del mismo nombre y modelo). Con «Perfil guardado», es el que se carga. En los diálogos, cualquier perfil guardado se usa con `[voz: nombre]…[/voz]`.*
# @markdown ---
# @markdown **Preparación del audio**
limpiar_audio = True  # @param {type:"boolean"}
# @markdown *Recorta los silencios del inicio y del final, quita retumbes graves y nivela el volumen.*
reducir_ruido = False  # @param {type:"boolean"}
# @markdown *Actívalo solo si se oye un ruido de fondo constante (ventilador, aire acondicionado).*
segundos_por_archivo = 60  # @param {type:"slider", min:6, max:120, step:2}
# @markdown *Segundos que se analizan de cada audio; entre 30 y 60 suele dar el mejor parecido.*

"""Celda 4 · Voz: grabación (Drive o tu computadora) → FFmpeg (WAV limpio) → perfil de voz en la GPU, que se puede
guardar en Drive y volver a cargar; registro de voces para los diálogos."""

import shutil
from collections.abc import Sequence
from pathlib import Path
from typing import Any

if globals().get("MOTOR") is None:
    raise RuntimeError("Ejecuta primero las celdas 1 a 3.")

SEGUNDOS_MINIMOS = 3.0
SEGUNDOS_RECOMENDADOS = 15.0
MAXIMO_ARCHIVOS = 12
SUBIR = "Subir desde mi computadora"
PERFIL = "Perfil guardado o voz incluida"
EXTENSIONES_PERFIL = (".xtts.npz", ".chatterbox.pt")


# ── Grabaciones ──


def audios_desde_drive(entrada: str) -> list[Path]:
    """El audio indicado en Drive, o los audios de la carpeta indicada."""
    montar_drive()
    return listar_medios(resolver_en_drive(entrada, extensiones=EXTENSIONES_MEDIOS, incluir_carpetas=True), MAXIMO_ARCHIVOS)


def audios_desde_computadora() -> list[Path]:
    """Muestra el botón de Colab para elegir uno o varios audios o videos de tu computadora (sin pasar por Drive)."""
    from google.colab import files

    carpeta = preparar_carpeta(CARPETA_TRABAJO / "subidas", vaciar=True)
    print("⬆️ Elige tu audio con el botón de abajo (puedes elegir varios)…")
    subidos = sorted(Path(nombre) for nombre in files.upload(target_dir=str(carpeta)))
    audios = [ruta for ruta in subidos if ruta.suffix.casefold() in EXTENSIONES_MEDIOS]
    if not audios:
        raise ErrorUsuario("No subiste ningún audio o video (WAV, MP3, M4A, OPUS, MP4…). Ejecuta la celda otra vez.")
    if len(audios) > MAXIMO_ARCHIVOS:
        print(f"ℹ️ Subiste {len(audios)} archivos; uso los primeros {MAXIMO_ARCHIVOS}.")
    return audios[:MAXIMO_ARCHIVOS]


def preparar_referencias(
    origenes: Sequence[Path], carpeta: Path, *, filtros: str, max_segundos: float
) -> list[tuple[Path, float]]:
    """Convierte cada audio o video en un WAV listo para el modelo; devuelve (ruta, segundos útiles)."""
    preparadas = []
    for indice, origen in enumerate(origenes, start=1):
        destino = carpeta / f"referencia_{indice:02d}.wav"
        preparar_referencia(origen, destino, filtros=filtros, max_segundos=max_segundos, frecuencia=MOTOR.frecuencia_referencia)
        muestras, frecuencia = leer_wav(destino)
        segundos = muestras.size / frecuencia
        if segundos < 0.5:
            print(f"   ⚠️ {origen.name}: casi no tiene voz; lo omito.")
            continue
        print(f"   🎧 {origen.name} → {segundos:.1f} s útiles")
        preparadas.append((destino, segundos))
    return preparadas


def crear_perfil(archivos: Sequence[Path], *, limpiar: bool, reducir: bool, segundos: float, nombre: str) -> PerfilVoz:
    """Prepara los audios y calcula la huella de la voz con el modelo cargado."""
    carpeta = preparar_carpeta(CARPETA_TRABAJO / "referencias", vaciar=True)
    print(f"🎚️ Preparando {len(archivos)} archivo(s)…")
    preparadas = preparar_referencias(
        archivos, carpeta, filtros=filtros_referencia(limpiar, reducir), max_segundos=segundos
    )
    total = sum(s for _, s in preparadas)
    if total < SEGUNDOS_MINIMOS:
        raise ErrorUsuario(f"Solo hay {total:.1f} s de voz útil. Usa una grabación de al menos 10 s (ideal 30–60 s).")
    if total < SEGUNDOS_RECOMENDADOS:
        print(f"⚠️ {total:.1f} s es poco: el parecido mejora mucho con 30–60 s de grabación.")
    voz = MOTOR.crear_voz([ruta for ruta, _ in preparadas], segundos, nombre=nombre)
    print(f"✅ Voz lista: {len(preparadas)} archivo(s), {formatear_duracion(total)} analizados. Así suena la referencia:")
    mostrar_reproductor(preparadas[0][0])
    return voz


# ── Perfiles guardados en Drive ──


def nombre_de_perfil(texto: str) -> str:
    return nombre_seguro(" ".join(texto.split()), por_defecto="mi_voz")


def perfiles_guardados(extension: str) -> dict[str, Path]:
    """{clave del nombre: archivo} de los perfiles de un modelo en «Clonador de voz/voces»."""
    carpeta = carpeta_del_proyecto("voces")
    return {
        clave_nombre(ruta.name[: -len(extension)]): ruta
        for ruta in sorted(carpeta.iterdir())
        if ruta.is_file() and ruta.name.casefold().endswith(extension)
    }


def buscar_perfil(nombre: str) -> Path | None:
    return perfiles_guardados(MOTOR.extension_perfil).get(clave_nombre(nombre_de_perfil(nombre)))


def guardar_perfil_en_drive(voz: PerfilVoz) -> None:
    """Escribe el perfil en Colab y lo copia a Drive; si Drive falla, la voz igual queda lista para esta sesión."""
    local = preparar_carpeta(CARPETA_TRABAJO / "perfiles") / f"{nombre_de_perfil(voz.nombre)}{MOTOR.extension_perfil}"
    MOTOR.guardar_voz(voz, local)
    try:
        destino = carpeta_del_proyecto("voces") / local.name
        existia = destino.exists()
        shutil.copyfile(local, destino)
    except DriveNoDisponible as error:
        print(f"⚠️ No se pudo guardar el perfil en Drive ({error.detalle}); la voz igual quedó lista para esta sesión.")
        return
    print(f"💾 Perfil «{voz.nombre}» guardado en {describir_ruta(destino)}" + (" (reemplazó al anterior)." if existia else "."))


def cargar_perfil(ruta: Path, nombre: str) -> PerfilVoz:
    voz, meta = MOTOR.cargar_voz(ruta)
    if meta.get("motor") != MOTOR.identificador:
        print(
            f"⚠️ El perfil «{nombre}» se creó con «{meta.get('motor')}» y ahora usas «{MOTOR.identificador}»: puede "
            "sonar menos parecido. Si es así, créalo otra vez desde la grabación."
        )
    detalle = f"{formatear_duracion(float(meta.get('segundos', 0.0)))} de referencia" if meta.get("segundos") else ""
    print(f"✅ Voz «{nombre}» cargada de {describir_ruta(ruta)}" + (f" ({detalle}, creada el {meta.get('creado')})." if detalle else "."))
    return voz


def no_encontrada(nombre: str) -> ErrorUsuario:
    """Error que lista lo que sí hay: perfiles de este modelo, perfiles de otro modelo y voces incluidas."""
    propios = sorted(ruta.name[: -len(MOTOR.extension_perfil)] for ruta in perfiles_guardados(MOTOR.extension_perfil).values())
    otros = [
        ruta.name
        for extension in EXTENSIONES_PERFIL
        if extension != MOTOR.extension_perfil
        for ruta in perfiles_guardados(extension).values()
        if clave_nombre(ruta.name[: -len(extension)]) == clave_nombre(nombre_de_perfil(nombre))
    ]
    partes = [f"No encontré la voz «{nombre}»."]
    if otros:
        partes.append(f"Existe «{otros[0]}», pero es de otro modelo: créala otra vez desde la grabación con {MOTOR.nombre}.")
    partes.append(f"Perfiles de {MOTOR.nombre} en Mi unidad/{CARPETA_PROYECTO}/voces: {', '.join(propios) or 'ninguno todavía'}.")
    incluidas = MOTOR.voces_incluidas()
    if incluidas:
        partes.append(f"Voces incluidas: {', '.join(incluidas[:6])}… (la lista completa sale al ejecutar esta celda).")
    return ErrorUsuario(" ".join(partes))


class RegistroDeVoces:
    """Resuelve los nombres de [voz: …]: la voz principal (celda 4), las voces incluidas en el modelo y tus perfiles
    guardados en Drive, en ese orden. Cada voz se prepara una sola vez por sesión."""

    def __init__(self, motor: Any, principal: PerfilVoz) -> None:
        self.motor = motor
        self.principal = principal
        self._incluidas = {clave_nombre(nombre): nombre for nombre in motor.voces_incluidas()}
        self._preparadas: dict[str, PerfilVoz] = {}

    def es_principal(self, nombre: str) -> bool:
        clave = clave_nombre(" ".join(nombre.split()))
        return clave in ("", "principal") or clave == clave_nombre(self.principal.nombre)

    def obtener(self, nombre: str) -> PerfilVoz:
        if self.es_principal(nombre):
            return self.principal
        clave = clave_nombre(" ".join(nombre.split()))
        if clave not in self._preparadas:
            self._preparadas[clave] = self._preparar(nombre, clave)
        return self._preparadas[clave]

    def _preparar(self, nombre: str, clave: str) -> PerfilVoz:
        if clave in self._incluidas:
            return self.motor.voz_incluida(self._incluidas[clave])
        ruta = buscar_perfil(nombre)
        if ruta is None:
            raise no_encontrada(nombre)
        return cargar_perfil(ruta, nombre)


def voz_guardada_o_incluida(nombre: str) -> PerfilVoz:
    """«Perfil guardado o voz incluida»: primero tus perfiles de Drive; si no hay, una voz incluida del modelo."""
    limpio = " ".join(nombre.split())
    if not limpio:
        raise ErrorUsuario("Escribe en «nombre_del_perfil» el perfil guardado o la voz incluida que quieres usar.")
    incluidas = {clave_nombre(voz): voz for voz in MOTOR.voces_incluidas()}
    try:
        ruta = buscar_perfil(limpio)
    except DriveNoDisponible:
        if clave_nombre(limpio) not in incluidas:
            raise
        ruta = None
    if ruta is not None:
        return cargar_perfil(ruta, limpio)
    if clave_nombre(limpio) in incluidas:
        print(f"✅ Voz incluida «{incluidas[clave_nombre(limpio)]}» lista.")
        return MOTOR.voz_incluida(incluidas[clave_nombre(limpio)])
    raise no_encontrada(limpio)


def mostrar_voces_disponibles() -> None:
    incluidas = MOTOR.voces_incluidas()
    if incluidas:
        print(f"🎭 Voces incluidas para diálogos ([voz: Nombre]…[/voz]): {', '.join(incluidas)}.")


# ─── Ejecución ───
VOZ = VOCES = None  # si algo falla, la celda 6 no usa una voz vieja
if origen_de_la_voz == PERFIL:
    VOZ = voz_guardada_o_incluida(nombre_del_perfil)
else:
    VOZ = crear_perfil(
        audios_desde_computadora() if origen_de_la_voz == SUBIR else audios_desde_drive(audio_en_drive),
        limpiar=limpiar_audio,
        reducir=reducir_ruido,
        segundos=segundos_por_archivo,
        nombre=" ".join(nombre_del_perfil.split()) or "mi_voz",
    )
    if guardar_perfil:
        guardar_perfil_en_drive(VOZ)
VOCES = RegistroDeVoces(MOTOR, VOZ)
mostrar_voces_disponibles()

In [ ]:
# @title 5 · Texto y pronunciación { display-mode: "form" }
fuente_del_texto = "Cuadro de texto"  # @param ["Cuadro de texto", "Archivo .txt de Drive"]
# @markdown *El cuadro aparece debajo al ejecutar esta celda: admite varios párrafos (cada salto de línea es un párrafo) y conserva lo que escribes mientras dure la sesión. Para textos que quieras guardar, usa un `.txt` de tu Drive:*
archivo_de_texto = ""  # @param {type:"string"}
idioma = "Español"  # @param ["Español", "English", "Português", "Français", "Italiano", "Deutsch", "Nederlands", "Polski", "Türkçe", "Русский", "Čeština", "Magyar", "العربية", "हिन्दी"]
# @markdown *Marcas en el texto: silencios `[pausa]` (1 s) o `[pausa 2.5]`; otro idioma con tu misma voz `[en]Where is the library?[/en]` (también `[pt]`, `[fr]`, `[it]`, `[de]`…); otra voz, para diálogos, `[voz: Ana Florence]¿Y tú qué opinas?[/voz]` (una voz incluida del modelo o un perfil guardado en la celda 4).*
modo_lote = False  # @param {type:"boolean"}
# @markdown *Lote: cada bloque separado por una línea con `---` se convierte en un audio aparte (un capítulo o una lección por archivo). Si la primera línea de un bloque es `# Título`, da nombre al archivo y no se lee.*
# @markdown ---
# @markdown **Pronunciación**
leer_montos_y_cifras = True  # @param {type:"boolean"}
# @markdown *Solo en español: «S/ 1,500.50» → «mil quinientos soles con cincuenta céntimos», «US$ 21» → «veintiún dólares» y «1,500» → «mil quinientos» (sin esto, el modelo lee «uno punto cinco»).*
pronunciacion = ""  # @param {type:"string"}
# @markdown *Reglas `texto = cómo se lee`, separadas por `;`. Ej.: `DNI = de ene i; Jr. = jirón; Av. = avenida`.*
archivo_de_pronunciacion = ""  # @param {type:"string"}
# @markdown *Para listas largas: un `.txt` en tu Drive con una regla por línea (las que empiezan con `#` son comentarios). Las reglas del cuadro de arriba tienen prioridad.*

"""Celda 5 · Texto: cuadro de varias líneas (o .txt de Drive), idioma, modo lote y pronunciación. Aquí no se genera
nada: la celda 6 lee el texto al ejecutarse, así que puedes editarlo y volver a generar sin repetir esta celda."""

from dataclasses import dataclass
from pathlib import Path
from typing import Any

if "dividir_texto" not in globals():
    raise RuntimeError("Ejecuta primero las celdas 1, 2a, 2b y 2c.")

CUADRO_DE_TEXTO = "Cuadro de texto"
TEXTO_DE_EJEMPLO = (
    "Hola, esta es una prueba de mi voz clonada. Puedo leer cualquier texto que escriba aquí, "
    "con pausas naturales entre frases."
)
BORRADOR = CARPETA_TRABAJO / "borrador.txt"  # sobrevive a un reinicio de la sesión (no a borrar el entorno)
CARACTERES_POR_SEGUNDO = 14.0  # ritmo típico de lectura, solo para estimar la duración


@dataclass(frozen=True)
class PedidoTexto:
    """Qué leer y cómo. El texto se lee al generar, así la celda 6 usa lo último que escribiste en el cuadro."""

    fuente: str
    archivo: str
    idioma: str
    lote: bool
    cifras: bool
    reglas: str
    archivo_reglas: str
    cuadro: Any = None

    def leer(self) -> str:
        """El contenido del cuadro o el del .txt de Drive."""
        if self.fuente == CUADRO_DE_TEXTO:
            return str(self.cuadro.value)
        montar_drive()
        ruta = resolver_en_drive(self.archivo, extensiones=EXTENSIONES_TEXTO, incluir_carpetas=False)
        contenido = leer_texto_de_archivo(ruta)
        print(f"📄 Texto de {describir_ruta(ruta)}: {len(contenido)} caracteres.")
        return contenido

    def pronunciador(self) -> Pronunciador | None:
        """Reglas del .txt de Drive (si se indicó) y después las del formulario, que tienen prioridad."""
        fuente = ""
        if self.archivo_reglas.strip():
            montar_drive()
            ruta = resolver_en_drive(self.archivo_reglas, extensiones=EXTENSIONES_TEXTO, incluir_carpetas=False)
            fuente = leer_texto_de_archivo(ruta)
        reglas, avisos = leer_reglas(f"{fuente}\n{self.reglas}")
        for aviso in avisos:
            print(f"⚠️ {aviso}")
        return Pronunciador(reglas) if reglas else None


def texto_inicial() -> str:
    """Lo que había en el cuadro (si la celda ya se ejecutó), el borrador guardado o un ejemplo."""
    anterior = globals().get("CUADRO")
    if anterior is not None:
        return str(anterior.value)
    if BORRADOR.is_file():
        return BORRADOR.read_text(encoding="utf-8")
    return TEXTO_DE_EJEMPLO


def crear_cuadro(inicial: str) -> Any:
    """Cuadro de texto de varias líneas (ipywidgets) con un contador; cada cambio se guarda como borrador."""
    import ipywidgets as widgets
    from IPython.display import display

    cuadro = widgets.Textarea(
        value=inicial,
        placeholder="Escribe o pega aquí el texto…",
        layout=widgets.Layout(width="100%", height="240px"),
    )
    contador = widgets.Label()

    def al_cambiar(_cambio: Any = None) -> None:
        texto = cuadro.value
        contador.value = (
            f"{len(texto)} caracteres · unos {formatear_duracion(len(texto) / CARACTERES_POR_SEGUNDO)} de audio"
        )
        preparar_carpeta(BORRADOR.parent)
        BORRADOR.write_text(texto, encoding="utf-8")

    cuadro.observe(al_cambiar, names="value")
    al_cambiar()
    display(widgets.VBox([cuadro, contador]))
    return cuadro


def preparar_pedido(fuente: str, archivo: str, nombre_idioma: str, lote: bool, cifras: bool, reglas: str, archivo_reglas: str) -> PedidoTexto:
    if fuente != CUADRO_DE_TEXTO and not archivo.strip():
        raise ErrorUsuario("Escribe el nombre del .txt en «archivo_de_texto» o elige «Cuadro de texto».")
    en_cuadro = fuente == CUADRO_DE_TEXTO
    donde = "el cuadro de abajo" if en_cuadro else f"«{archivo.strip()}» (se lee de tu Drive al generar)"
    modo = " · modo lote: un audio por bloque separado con ---" if lote else ""
    print(f"✍️ Texto: {donde} · idioma {nombre_idioma}{modo}. Cuando esté listo, ejecuta la celda 6.")
    cuadro = crear_cuadro(texto_inicial()) if en_cuadro else None  # se muestra debajo del mensaje
    return PedidoTexto(fuente, archivo, IDIOMAS[nombre_idioma], lote, cifras, reglas, archivo_reglas, cuadro)


# ─── Ejecución ───
PEDIDO_TEXTO = None  # si algo falla, la celda 6 no usa un pedido viejo
PEDIDO_TEXTO = preparar_pedido(
    fuente_del_texto,
    archivo_de_texto,
    idioma,
    modo_lote,
    leer_montos_y_cifras,
    pronunciacion,
    archivo_de_pronunciacion,
)
CUADRO = PEDIDO_TEXTO.cuadro

In [ ]:
# @title 6 · Generar el audio { display-mode: "form" }
# @markdown **Estilo**
velocidad = 1.0  # @param {type:"slider", min:0.6, max:1.6, step:0.05}
temperatura = 0.75  # @param {type:"slider", min:0.1, max:1.0, step:0.05}
# @markdown *Más baja: más estable y monótona. Más alta: más expresiva, pero con más riesgo de errores.*
pausa_entre_frases = 0.3  # @param {type:"slider", min:0.0, max:1.5, step:0.05}
# @markdown ---
# @markdown **Tomas y control de calidad**
tomas = 1  # @param {type:"slider", min:1, max:4, step:1}
# @markdown *Versiones distintas del mismo texto. Se ordenan por calidad y parecido con tu voz, y eliges una en la celda 7 (en modo lote se guarda sola la mejor).*
semilla = -1  # @param {type:"integer"}
# @markdown *`-1` = al azar. Cada toma imprime su semilla: cópiala aquí (con `tomas` en 1) para repetir ese resultado.*
control_de_calidad = True  # @param {type:"boolean"}
# @markdown *Revisa cada fragmento (ritmo y silencios a mitad de frase) y lo vuelve a generar hasta 2 veces si falla.*
verificar_con_whisper = False  # @param {type:"boolean"}
# @markdown *Además, transcribe cada fragmento con Whisper y lo compara con el texto: detecta palabras omitidas o balbuceos. Más lento; la primera vez instala y descarga Whisper (~1,6 GB).*
# @markdown ---
# @markdown **Salida**
nombre_salida = "mi_voz_clonada"  # @param {type:"string"}
formato = "mp3"  # @param ["mp3", "wav", "ogg"]
volumen = "Parejo: -16 LUFS (voz)"  # @param ["Parejo: -16 LUFS (voz)", "Alto: -14 LUFS (redes)", "Suave: -20 LUFS", "Sin nivelar"]
# @markdown *Todas las tomas y generaciones salen con el mismo volumen percibido, sin que los picos pasen de −1 dBFS.*
subtitulos = False  # @param {type:"boolean"}
# @markdown *Crea también un `.srt` sincronizado con el audio (para videos), con el texto tal como lo escribiste.*
musica_de_fondo = ""  # @param {type:"string"}
# @markdown *Opcional: nombre de un audio de tu Drive. Suena en bucle, entra antes de la voz y baja sola mientras hablas.*
volumen_musica = -18  # @param {type:"slider", min:-30, max:-6, step:1}
# @markdown *Decibeles por debajo de la voz en las pausas; mientras hablas baja 12 dB más.*
guardar_en_drive = True  # @param {type:"boolean"}
carpeta_en_drive = "Clonador de voz"  # @param {type:"string"}
descargar_al_terminar = False  # @param {type:"boolean"}
# @markdown *Con varias tomas, el guardado en Drive y la descarga se hacen desde la celda 7, con la toma elegida.*
# @markdown ---
# @markdown **Solo Chatterbox**
exageracion = 0.5  # @param {type:"slider", min:0.25, max:2.0, step:0.05}
# @markdown *Expresividad: 0.5 es neutra; más alta, más emoción (y un habla algo más rápida).*
ritmo = 0.5  # @param {type:"slider", min:0.0, max:1.0, step:0.05}
# @markdown *Apego al estilo de tu grabación. Con exageración alta, bájalo (~0.3) para un ritmo más pausado.*
# @markdown ---
# @markdown **Avanzado: muestreo de XTTS-v2**
top_p = 0.85  # @param {type:"slider", min:0.1, max:1.0, step:0.05}
top_k = 50  # @param {type:"slider", min:1, max:100, step:1}
penalizacion_repeticion = 5.0  # @param {type:"slider", min:1.0, max:15.0, step:0.5}

"""Celda 6 · Generación: texto → tramos (idioma y voz) → fragmentos → modelo con control de calidad → tomas ordenadas
por calidad y parecido → música, volumen y subtítulos → archivos en Colab y en Drive. En modo lote, un audio por
bloque."""

import random
import shutil
import time
from collections import Counter, defaultdict
from collections.abc import Sequence
from dataclasses import dataclass, field, replace
from pathlib import Path

import numpy as np

if any(globals().get(nombre) is None for nombre in ("MOTOR", "VOZ", "VOCES")):
    raise RuntimeError("Ejecuta primero las celdas 1 a 4.")
if globals().get("PEDIDO_TEXTO") is None:
    raise RuntimeError("Ejecuta primero la celda 5 · Texto y pronunciación.")

VOLUMENES = {"Parejo: -16 LUFS (voz)": -16.0, "Alto: -14 LUFS (redes)": -14.0, "Suave: -20 LUFS": -20.0, "Sin nivelar": None}
CONVERSORES = {"mp3": convertir_a_mp3, "ogg": convertir_a_ogg}  # «wav» se escribe directo
MARGEN_LIMITE = 10  # caracteres por debajo del límite del modelo, por seguridad
REINTENTOS_POR_MITADES = 2  # un fragmento que el modelo rechaza se parte en mitades hasta 2 veces
SEMILLA_MAXIMA = 2**31 - 1
REINTENTOS_DE_CALIDAD = 2
RITMO_NOMINAL = 14.0  # caracteres por segundo a velocidad 1, hasta tener 3 fragmentos aceptados de esa voz e idioma
RITMO_MINIMO, RITMO_MAXIMO = 0.55, 1.9  # fuera de ese rango (relativo al habitual), el fragmento es sospechoso
CARACTERES_PARA_RITMO = 12  # en textos más cortos, el ritmo no dice nada
SILENCIO_INTERNO_MAXIMO = 1.2  # segundos callado a mitad de un fragmento
CER_MAXIMO = 0.30  # tasa de error por caracteres tolerada entre lo pedido y lo que transcribe Whisper
SEGUNDOS_PARA_PARECIDO = 40.0  # audio de la toma con que se mide su parecido con tu voz
FRECUENCIA_MEZCLA = 48000  # con música, la mezcla se hace a 48 kHz para no recortar sus agudos
MAXIMO_REPRODUCTORES_LOTE = 6


@dataclass(frozen=True)
class OpcionesTexto:
    pausa_frase: float
    cifras: bool
    pronunciador: Pronunciador | None


@dataclass(frozen=True)
class OpcionesCalidad:
    activo: bool
    whisper: bool


@dataclass(frozen=True)
class OpcionesSalida:
    nombre: str
    formato: str
    sonoridad: float | None  # LUFS objetivo; None = solo normalizar el pico
    subtitulos: bool
    musica: str  # audio de Drive para el fondo, o vacío
    volumen_musica: float
    guardar_en_drive: bool
    carpeta_en_drive: str
    descargar: bool


@dataclass(frozen=True)
class Revision:
    """Resultado del control de calidad de un fragmento; `gravedad` 0 = sin problemas."""

    problemas: tuple[str, ...] = ()
    gravedad: float = 0.0


@dataclass(frozen=True)
class Fragmento:
    onda: np.ndarray
    segmento: Segmento
    revision: Revision
    intentos: int  # cuántas veces se sintetizó (1 = pasó a la primera)


@dataclass
class Toma:
    """Un resultado completo; `en_drive` se llena al copiarlo a Drive (en la celda 6 o en la 7)."""

    numero: int
    semilla: int
    archivo: Path
    subtitulos: Path | None = None
    parecido: float | None = None
    avisos: int = 0
    en_drive: Path | None = None


@dataclass
class Generacion:
    nombre: str  # nombre de salida (o título del bloque, en modo lote)
    tomas: list[Toma]
    salida: OpcionesSalida
    recomendada: int = 1
    lote: bool = False


# ── Texto ──


def preparar_segmentos(texto: str, principal: str, opciones: OpcionesTexto, disponibles: frozenset[str]) -> list[Segmento]:
    """Separa el texto en tramos por idioma y voz y cada tramo en frases visibles (las de los subtítulos). A cada
    frase le adapta montos y cifras (si es español) y le aplica la pronunciación (si es el idioma principal); si
    eso la alarga más que el límite del modelo, la vuelve a partir en trozos del mismo grupo."""
    tramos, avisos = dividir_en_tramos(texto, principal, disponibles)
    for aviso in avisos:
        print(f"⚠️ {aviso}")
    segmentos: list[Segmento] = []
    cifras = reemplazos = grupo = 0
    anterior = ""
    for tramo in tramos:
        limite = MOTOR.limite_caracteres(tramo.idioma) - MARGEN_LIMITE
        nuevos: list[Segmento] = []
        for visible in dividir_texto(tramo.texto, limite, opciones.pausa_frase):
            lectura = visible.texto
            if opciones.cifras and tramo.idioma == "es":
                lectura, cambios = normalizar_cifras(lectura)
                cifras += cambios
            if opciones.pronunciador is not None and tramo.idioma == principal:
                lectura, cambios = opciones.pronunciador.aplicar(lectura)
                reemplazos += cambios
            piezas = partir_lectura(lectura, limite)
            if not piezas:  # la regla dejó la frase sin nada que leer: su pausa pasa a la anterior
                if nuevos:
                    nuevos[-1] = replace(nuevos[-1], pausa=nuevos[-1].pausa + visible.pausa)
                continue
            for numero, pieza in enumerate(piezas, start=1):
                pausa = visible.pausa if numero == len(piezas) else opciones.pausa_frase
                nuevos.append(Segmento(pieza, pausa, tramo.idioma, tramo.voz, visible.texto, grupo))
            grupo += 1
        if segmentos:  # frontera entre tramos: pausa de frase o de párrafo, más los [pausa] que haya al inicio
            frontera = opciones.pausa_frase
            if hay_salto_de_parrafo(anterior, tramo.texto):
                frontera = pausa_de_parrafo(opciones.pausa_frase)
            pausa = max(segmentos[-1].pausa, frontera) if nuevos else segmentos[-1].pausa
            segmentos[-1] = replace(segmentos[-1], pausa=pausa + pausa_inicial(tramo.texto))
        segmentos.extend(nuevos)
        anterior = tramo.texto if nuevos else anterior + tramo.texto
    if not segmentos:
        raise ErrorUsuario("No hay texto que leer: escribe algo en el cuadro de la celda 5 o indica un archivo .txt.")
    if cifras:
        print(f"💲 Montos y cifras: {cifras} adaptado(s) para que se lean bien.")
    if opciones.pronunciador is not None:
        print(f"🔤 Pronunciación: {opciones.pronunciador.cantidad} regla(s) · {reemplazos} reemplazo(s) en el texto.")
    por_idioma = Counter(segmento.idioma for segmento in segmentos)
    if len(por_idioma) > 1:
        print("🌐 Idiomas: " + " · ".join(f"{codigo} ({n} fragmento(s))" for codigo, n in por_idioma.items()))
    return segmentos


def voces_del_texto(segmentos: Sequence[Segmento]) -> dict[str, PerfilVoz]:
    """Prepara antes de sintetizar todas las voces que aparecen: si falta una, el error sale de inmediato."""
    voces: dict[str, PerfilVoz] = {}
    for segmento in segmentos:
        if segmento.voz not in voces:
            voces[segmento.voz] = VOCES.obtener(segmento.voz)
    otras = [nombre for nombre in voces if not VOCES.es_principal(nombre)]
    if otras:
        print("🎭 Voces: la tuya, " + ", ".join(otras) + ".")
    return voces


# ── Control de calidad ──


class Revisor:
    """Revisa cada fragmento: ritmo frente al habitual de esa voz e idioma (o uno nominal), silencios a mitad de
    frase y, si se pide, lo que entiende Whisper."""

    def __init__(self, opciones: OpcionesCalidad, velocidad: float) -> None:
        self.opciones = opciones
        self.nominal = RITMO_NOMINAL * float(velocidad)
        self._ritmos: dict[tuple[str, str], list[float]] = defaultdict(list)

    def _ritmo_habitual(self, segmento: Segmento) -> float:
        ritmos = self._ritmos[(segmento.voz, segmento.idioma)]
        return float(np.median(ritmos)) if len(ritmos) >= 3 else self.nominal

    def revisar(self, onda: np.ndarray, segmento: Segmento, frecuencia: int) -> Revision:
        problemas: list[str] = []
        gravedad = 0.0
        segundos = onda.size / frecuencia
        if len(segmento.texto) >= CARACTERES_PARA_RITMO:
            relativo = len(segmento.texto) / max(segundos, 1e-3) / self._ritmo_habitual(segmento)
            if relativo < RITMO_MINIMO:
                problemas.append(f"dura demasiado para su texto ({segundos:.1f} s)")
                gravedad += RITMO_MINIMO / relativo
            elif relativo > RITMO_MAXIMO:
                problemas.append(f"dura muy poco para su texto ({segundos:.1f} s)")
                gravedad += relativo / RITMO_MAXIMO
        silencio = silencio_interno_maximo(onda, frecuencia)
        if silencio > SILENCIO_INTERNO_MAXIMO:
            problemas.append(f"silencio de {silencio:.1f} s a mitad de frase")
            gravedad += silencio / SILENCIO_INTERNO_MAXIMO
        if self.opciones.whisper:
            error = tasa_error_caracteres(segmento.texto, transcribir(onda, frecuencia, segmento.idioma), segmento.idioma)
            if error > CER_MAXIMO:
                problemas.append(f"Whisper entendió otra cosa ({error:.0%} de error)")
                gravedad += error / CER_MAXIMO
        return Revision(tuple(problemas), gravedad)

    def aceptar(self, onda: np.ndarray, segmento: Segmento, frecuencia: int) -> None:
        if len(segmento.texto) >= CARACTERES_PARA_RITMO:
            self._ritmos[(segmento.voz, segmento.idioma)].append(len(segmento.texto) / (onda.size / frecuencia))


# ── Síntesis ──


def sintetizar_tolerante(texto: str, voz: PerfilVoz, ajustes: AjustesSintesis, profundidad: int = 0) -> np.ndarray | None:
    """Sintetiza un fragmento; si el modelo lo rechaza, reintenta por mitades en vez de perder todo el audio.

    Ejemplo típico: XTTS admite menos de 400 tokens y los números se expanden a palabras («1999» → «mil
    novecientos noventa y nueve») después de dividir el texto por caracteres.
    """
    try:
        return MOTOR.sintetizar(texto, voz, ajustes)
    except Exception as error:  # noqa: BLE001 — se informa y se continúa con el resto del texto
        mitades = partir_por_la_mitad(texto) if profundidad < REINTENTOS_POR_MITADES else None
        if mitades is None:
            print(f"\n⚠️ Se omitió «{texto[:60]}» ({type(error).__name__}: {error})")
            return None
        partes = [sintetizar_tolerante(mitad, voz, ajustes, profundidad + 1) for mitad in mitades]
        partes = [parte for parte in partes if parte is not None and parte.size]
        return np.concatenate(partes) if partes else None


def semilla_derivada(semilla: int, indice: int, intento: int) -> int:
    """Semilla de un reintento: distinta para cada fragmento e intento, pero repetible con la semilla de la toma."""
    return (semilla * 1_000_003 + (indice + 1) * 7_919 + intento) % SEMILLA_MAXIMA


def sintetizar_fragmento(
    indice: int, segmento: Segmento, voz: PerfilVoz, ajustes: AjustesSintesis, revisor: Revisor, semilla: int
) -> Fragmento | None:
    """Sintetiza un fragmento y, con el control de calidad activo, lo regenera hasta que pase o se acaben los
    intentos; si ninguno pasa, se queda con el menos malo."""
    maximo = 1 + (REINTENTOS_DE_CALIDAD if revisor.opciones.activo else 0)
    mejor: Fragmento | None = None
    realizados = 0
    for intento in range(maximo):
        if intento:
            MOTOR.fijar_semilla(semilla_derivada(semilla, indice, intento))
        realizados += 1
        onda = sintetizar_tolerante(segmento.texto, voz, ajustes)
        onda = recortar_silencios(onda, MOTOR.frecuencia) if onda is not None else None
        if onda is None or onda.size == 0:
            continue
        revision = revisor.revisar(onda, segmento, MOTOR.frecuencia) if revisor.opciones.activo else Revision()
        if mejor is None or revision.gravedad < mejor.revision.gravedad:
            mejor = Fragmento(aplicar_fundidos(onda, MOTOR.frecuencia), segmento, revision, 0)
        if not revision.problemas:
            break
    if mejor is None:
        return None
    if not mejor.revision.problemas:
        revisor.aceptar(mejor.onda, segmento, MOTOR.frecuencia)
    return replace(mejor, intentos=realizados)


def sintetizar_segmentos(
    segmentos: Sequence[Segmento], voces: dict[str, PerfilVoz], ajustes: AjustesSintesis, revisor: Revisor, semilla: int
) -> list[Fragmento]:
    fragmentos = []
    for indice, segmento in enumerate(segmentos):
        print(f"\r🗣️ Fragmento {indice + 1}/{len(segmentos)}…", end="", flush=True)
        ajustes_fragmento = ajustes if segmento.idioma in ("", ajustes.idioma) else replace(ajustes, idioma=segmento.idioma)
        fragmento = sintetizar_fragmento(indice, segmento, voces[segmento.voz], ajustes_fragmento, revisor, semilla)
        if fragmento is None:
            print(f"\n⚠️ El fragmento {indice + 1} quedó sin audio: «{segmento.texto[:60]}»")
            continue
        fragmentos.append(fragmento)
    print("\r", end="")
    return fragmentos


def informar_calidad(fragmentos: Sequence[Fragmento]) -> int:
    """Resume el control de calidad; devuelve cuántos fragmentos quedaron con avisos."""
    regenerados = sum(1 for fragmento in fragmentos if fragmento.intentos > 1)
    con_avisos = [fragmento for fragmento in fragmentos if fragmento.revision.problemas]
    if regenerados:
        print(f"   🔁 Control de calidad: {regenerados} fragmento(s) se regeneraron.")
    for fragmento in con_avisos[:5]:
        print(f"   ⚠️ «{fragmento.segmento.texto[:50]}…» {'; '.join(fragmento.revision.problemas)} (se usó el mejor de {fragmento.intentos} intentos)")
    if len(con_avisos) > 5:
        print(f"   ⚠️ …y {len(con_avisos) - 5} fragmento(s) más con avisos.")
    return len(con_avisos)


def muestra_repartida(ondas: Sequence[np.ndarray], maximo: int) -> np.ndarray:
    """Hasta `maximo` muestras tomadas de fragmentos repartidos por toda la toma (no solo del principio)."""
    total = sum(onda.size for onda in ondas)
    if total <= maximo:
        return np.concatenate(ondas)
    cantidad = max(1, round(len(ondas) * maximo / total))
    indices = np.unique(np.linspace(0, len(ondas) - 1, cantidad).round().astype(int))
    return np.concatenate([ondas[i] for i in indices])[:maximo]


def medir_parecido(fragmentos: Sequence[Fragmento], principal: PerfilVoz) -> float | None:
    """Similitud entre el timbre de la toma (solo los fragmentos con tu voz) y el de tu voz de referencia."""
    propios = [fragmento.onda for fragmento in fragmentos if VOCES.es_principal(fragmento.segmento.voz)]
    if not propios:
        return None
    muestra = muestra_repartida(propios, round(SEGUNDOS_PARA_PARECIDO * MOTOR.frecuencia))
    return parecido(MOTOR.huella(muestra, MOTOR.frecuencia), principal.huella)


# ── Salida ──


def cargar_musica(nombre: str) -> np.ndarray | None:
    """La música de fondo indicada (de Drive), en estéreo a la frecuencia de la mezcla."""
    if not nombre.strip():
        return None
    montar_drive()
    ruta = resolver_en_drive(nombre, extensiones=EXTENSIONES_MEDIOS, incluir_carpetas=False)
    musica = decodificar_audio(ruta, FRECUENCIA_MEZCLA, canales=2, max_segundos=20 * 60)
    print(f"🎵 Música de fondo: {describir_ruta(ruta)} ({formatear_duracion(musica.shape[0] / FRECUENCIA_MEZCLA)}, en bucle).")
    return musica


def subtitulos_de(fragmentos: Sequence[Fragmento], posiciones: Sequence[tuple[int, int]], frecuencia: int, desfase: float) -> list[tuple[float, float, str]]:
    """Un subtítulo por frase visible: del inicio de su primer trozo al final del último."""
    entradas: list[tuple[float, float, str]] = []
    grupo_anterior = None
    for fragmento, (inicio, fin) in zip(fragmentos, posiciones, strict=True):
        segmento = fragmento.segmento
        desde, hasta = inicio / frecuencia + desfase, fin / frecuencia + desfase
        if entradas and segmento.grupo == grupo_anterior:
            entradas[-1] = (entradas[-1][0], hasta, entradas[-1][2])
        else:
            entradas.append((desde, hasta, " ".join((segmento.visible or segmento.texto).split())))
        grupo_anterior = segmento.grupo
    return entradas


def exportar(audio: np.ndarray, frecuencia: int, nombre: str, formato: str, con_subtitulos: bool) -> Path:
    """Escribe el audio en /content/clonador_voz/salidas sin sobrescribir nada (ni su .srt, si lo habrá)."""
    carpeta = preparar_carpeta(CARPETA_TRABAJO / "salidas")
    extensiones = (f".{formato}", ".srt") if con_subtitulos else (f".{formato}",)
    destino = carpeta / f"{base_libre(carpeta, nombre_seguro(nombre), extensiones)}.{formato}"
    if formato == "wav":
        escribir_wav(destino, audio, frecuencia)
        return destino
    temporal = carpeta / "_temporal.wav"
    escribir_wav(temporal, audio, frecuencia)
    try:
        CONVERSORES[formato](temporal, destino)
    finally:
        temporal.unlink(missing_ok=True)
    return destino


def semillas_para(semilla: int, cantidad: int) -> list[int]:
    """La semilla indicada (o una al azar) y las consecutivas: cada toma se puede repetir por separado."""
    base = semilla if semilla >= 0 else random.randrange(1, SEMILLA_MAXIMA)
    return [(base + desplazamiento) % SEMILLA_MAXIMA for desplazamiento in range(cantidad)]


def generar_toma(
    numero: int,
    total: int,
    nombre: str,
    segmentos: Sequence[Segmento],
    voces: dict[str, PerfilVoz],
    ajustes: AjustesSintesis,
    calidad: OpcionesCalidad,
    semilla: int,
    salida: OpcionesSalida,
    musica: np.ndarray | None,
) -> Toma:
    """Sintetiza el texto completo con una semilla, lo mezcla, lo nivela y lo exporta como archivo en Colab."""
    MOTOR.fijar_semilla(semilla)
    inicio = time.perf_counter()
    fragmentos = sintetizar_segmentos(segmentos, voces, ajustes, Revisor(calidad, ajustes.velocidad), semilla)
    if not fragmentos:
        raise ErrorUsuario("El modelo no produjo audio. Prueba con otra semilla o revisa el texto.")
    audio, posiciones = ensamblar([f.onda for f in fragmentos], [f.segmento.pausa for f in fragmentos], MOTOR.frecuencia)
    similitud = medir_parecido(fragmentos, VOCES.principal)
    frecuencia, desfase = MOTOR.frecuencia, 0.0
    if musica is not None:
        frecuencia, desfase = FRECUENCIA_MEZCLA, ENTRADA_MUSICA_S
        audio = mezclar_con_musica(remuestrear(audio, MOTOR.frecuencia, frecuencia), musica, frecuencia, nivel_db=salida.volumen_musica)
    if salida.sonoridad is None:
        audio, volumen = normalizar_pico(audio), ""
    else:
        audio = nivelar_volumen(audio, frecuencia, salida.sonoridad)
        volumen = f" · {sonoridad_integrada(audio, frecuencia):.1f} LUFS"
    archivo = exportar(audio, frecuencia, f"{nombre}_toma{numero}" if total > 1 else nombre, salida.formato, salida.subtitulos)
    srt = None
    if salida.subtitulos:
        srt = archivo.with_suffix(".srt")
        srt.write_text(subtitulos_srt(subtitulos_de(fragmentos, posiciones, MOTOR.frecuencia, desfase)), encoding="utf-8")
    prefijo = f"Toma {numero}/{total} · " if total > 1 else ""
    detalle_parecido = f" · parecido {similitud:.3f}" if similitud is not None else ""
    print(
        f"✅ {prefijo}{len(fragmentos)} fragmento(s) · {formatear_duracion(audio.shape[0] / frecuencia)} de audio"
        f"{volumen} · generado en {time.perf_counter() - inicio:.1f} s · semilla {semilla}{detalle_parecido}"
    )
    if len(fragmentos) < len(segmentos):
        print(f"⚠️ {len(segmentos) - len(fragmentos)} fragmento(s) quedaron sin audio (ver avisos): revisa ese texto o cambia la semilla.")
    avisos = informar_calidad(fragmentos) if calidad.activo else 0
    return Toma(numero, semilla, archivo, srt, similitud, avisos)


def ordenar_tomas(tomas: Sequence[Toma]) -> list[Toma]:
    """Primero las que tienen menos fragmentos con avisos; entre ellas, la más parecida a tu voz."""
    return sorted(tomas, key=lambda t: (t.avisos, -(t.parecido if t.parecido is not None else 0.0), t.numero))


def copiar_a_drive(archivos: Sequence[Path], carpeta_texto: str, nombre: str) -> Path:
    """Copia el audio (y su .srt) a la carpeta indicada de Drive con el mismo nombre libre; devuelve la del audio."""
    montar_drive()
    carpeta = preparar_carpeta(ruta_en_mi_unidad(carpeta_texto.strip() or CARPETA_PROYECTO))
    base = base_libre(carpeta, nombre_seguro(nombre), tuple(archivo.suffix for archivo in archivos))
    destinos = [carpeta / f"{base}{archivo.suffix}" for archivo in archivos]
    for archivo, destino in zip(archivos, destinos, strict=True):
        shutil.copyfile(archivo, destino)
    return destinos[0]


def archivos_de(toma: Toma) -> list[Path]:
    return [toma.archivo] + ([toma.subtitulos] if toma.subtitulos is not None else [])


def guardar_toma_en_drive(toma: Toma, generacion: Generacion) -> Path:
    """Copia la toma a Drive con el nombre de salida (sin «_tomaN»); si ya se copió, no la duplica."""
    if toma.en_drive is None:
        toma.en_drive = copiar_a_drive(archivos_de(toma), generacion.salida.carpeta_en_drive, generacion.nombre)
        extra = " (y sus subtítulos .srt)" if toma.subtitulos is not None else ""
        print(f"💾 Guardado en {describir_ruta(toma.en_drive)}{extra}")
    else:
        print(f"ℹ️ Ya estaba guardada en {describir_ruta(toma.en_drive)}")
    return toma.en_drive


def intentar_guardar_en_drive(toma: Toma, generacion: Generacion) -> None:
    """Como guardar_toma_en_drive, pero si Drive no se conecta el audio ya generado no se pierde: queda en Colab."""
    try:
        guardar_toma_en_drive(toma, generacion)
    except DriveNoDisponible as error:
        print(
            f"⚠️ No se pudo guardar en Drive ({error.detalle}). El audio quedó en Colab: reintenta el guardado con la "
            "celda 7 (elige «1»; revisa antes la cuenta y los permisos de Drive) o márcala con «descargar_tambien»."
        )


def descargar(toma: Toma) -> None:
    from google.colab import files

    for archivo in archivos_de(toma):
        files.download(str(archivo))


def entregar(toma: Toma, generacion: Generacion, *, reproductor: bool) -> None:
    print(f"   Archivo en Colab: {toma.archivo}")
    if generacion.salida.guardar_en_drive:
        intentar_guardar_en_drive(toma, generacion)
    if reproductor:
        mostrar_reproductor(toma.archivo)
    if generacion.salida.descargar:
        descargar(toma)


# ── Orquestación ──


def generar_bloque(
    texto: str,
    nombre: str,
    ajustes: AjustesSintesis,
    opciones: OpcionesTexto,
    calidad: OpcionesCalidad,
    *,
    semilla: int,
    tomas: int,
    salida: OpcionesSalida,
    musica: np.ndarray | None,
    lote: bool = False,
    reproductor: bool = True,
) -> Generacion:
    """Una o varias tomas de un texto. Con una sola (o en lote, con la mejor) se entrega de inmediato; con varias,
    la elección queda para la celda 7."""
    segmentos = preparar_segmentos(texto, ajustes.idioma, opciones, MOTOR.idiomas)
    voces = voces_del_texto(segmentos)
    total = max(1, tomas)
    generacion = Generacion(nombre, [], salida, lote=lote)
    for numero, semilla_toma in enumerate(semillas_para(semilla, total), start=1):
        toma = generar_toma(numero, total, nombre, segmentos, voces, ajustes, calidad, semilla_toma, salida, musica)
        generacion.tomas.append(toma)
        if total > 1 and not lote:
            mostrar_reproductor(toma.archivo)
    mejor = ordenar_tomas(generacion.tomas)[0]
    generacion.recomendada = mejor.numero
    if total > 1:
        for toma in generacion.tomas:
            detalle = f"parecido {toma.parecido:.3f}" if toma.parecido is not None else "parecido sin medir"
            marca = "  ⭐ recomendada" if toma is mejor else ""
            print(f"   Toma {toma.numero}: {detalle} · {toma.avisos} fragmento(s) con avisos{marca}")
    if total > 1 and not lote:
        print(f"👉 Escucha las {total} tomas y guarda la mejor con la celda 7 (la recomendada es la {mejor.numero}).")
        return generacion
    entregar(mejor, generacion, reproductor=reproductor)
    return generacion


def generar_audio(
    pedido: PedidoTexto, ajustes: AjustesSintesis, calidad: OpcionesCalidad, *, pausa_frase: float, semilla: int, tomas: int, salida: OpcionesSalida
) -> list[Generacion]:
    """Lee el texto (cuadro o .txt) y genera un audio, o uno por bloque en modo lote."""
    if ajustes.idioma not in MOTOR.idiomas:
        raise ErrorUsuario(f"{MOTOR.nombre} no habla el idioma «{ajustes.idioma}»: elige otro en la celda 5.")
    texto = pedido.leer()
    opciones = OpcionesTexto(pausa_frase=pausa_frase, cifras=pedido.cifras, pronunciador=pedido.pronunciador())
    musica = cargar_musica(salida.musica)
    if not pedido.lote:
        return [generar_bloque(texto, salida.nombre, ajustes, opciones, calidad, semilla=semilla, tomas=tomas, salida=salida, musica=musica)]
    bloques = dividir_en_bloques(texto)
    if not bloques:
        raise ErrorUsuario("No hay bloques con texto: sepáralos con una línea «---».")
    print(f"📚 Modo lote: {len(bloques)} bloque(s).")
    generaciones, fallidos = [], []
    for indice, bloque in enumerate(bloques, start=1):
        nombre = nombre_seguro(bloque.titulo) if bloque.titulo else f"{nombre_seguro(salida.nombre)}_{indice:02d}"
        print(f"\n── Bloque {indice}/{len(bloques)} · «{nombre}» ──")
        try:
            generaciones.append(
                generar_bloque(
                    bloque.texto, nombre, ajustes, opciones, calidad, semilla=semilla, tomas=tomas, salida=salida,
                    musica=musica, lote=True, reproductor=indice <= MAXIMO_REPRODUCTORES_LOTE,
                )
            )
        except ErrorUsuario as error:  # un bloque con problemas no detiene a los demás
            print(f"❌ Bloque {indice} («{nombre}»): {error}")
            fallidos.append(nombre)
    if not generaciones:
        raise ErrorUsuario("Ningún bloque produjo audio (ver los mensajes de arriba).")
    print(f"\n✅ Lote terminado: {len(generaciones)} audio(s)" + (f"; con errores: {', '.join(fallidos)}." if fallidos else "."))
    if len(bloques) > MAXIMO_REPRODUCTORES_LOTE:
        print(f"ℹ️ Se muestran los reproductores de los primeros {MAXIMO_REPRODUCTORES_LOTE} bloques; todos los audios están guardados.")
    return generaciones


# ─── Ejecución ───
ULTIMO_LOTE = generar_audio(
    PEDIDO_TEXTO,
    AjustesSintesis(
        idioma=PEDIDO_TEXTO.idioma,
        velocidad=velocidad,
        temperatura=temperatura,
        top_p=top_p,
        top_k=int(top_k),
        penalizacion_repeticion=penalizacion_repeticion,
        exageracion=exageracion,
        cfg=ritmo,
    ),
    OpcionesCalidad(activo=control_de_calidad or verificar_con_whisper, whisper=verificar_con_whisper),
    pausa_frase=pausa_entre_frases,
    semilla=int(semilla),
    tomas=int(tomas),
    salida=OpcionesSalida(
        nombre=nombre_salida,
        formato=formato,
        sonoridad=VOLUMENES[volumen],
        subtitulos=subtitulos,
        musica=musica_de_fondo,
        volumen_musica=float(volumen_musica),
        guardar_en_drive=guardar_en_drive,
        carpeta_en_drive=carpeta_en_drive,
        descargar=descargar_al_terminar,
    ),
)
ULTIMA_GENERACION = ULTIMO_LOTE[-1]

In [ ]:
# @title 7 · Guardar la toma elegida { display-mode: "form" }
# @markdown Si en la celda 6 pediste varias **tomas**, elige aquí la mejor (la celda 6 recomienda una según el control de calidad y el parecido con tu voz). Se guarda en tu Drive con la carpeta y el nombre de salida de la celda 6, junto con sus subtítulos si los pediste.
toma_elegida = "—"  # @param ["—", "Recomendada", "1", "2", "3", "4"]
descargar_tambien = False  # @param {type:"boolean"}

"""Celda 7 · Elección: guarda en Drive (y descarga, si se pide) una toma de la última generación."""

if globals().get("ULTIMA_GENERACION") is None:
    raise RuntimeError("Ejecuta primero la celda 6.")

RECOMENDADA = "Recomendada"


def guardar_toma_elegida(eleccion: str, generacion: Generacion, *, descargar_archivo: bool) -> Toma | None:
    """Guarda la toma elegida. Con «—» solo orienta, para que «Ejecutar todas» no guarde nada por su cuenta."""
    cantidad = len(generacion.tomas)
    if generacion.lote:
        print(
            "ℹ️ En modo lote, la celda 6 ya guardó la mejor toma de cada bloque. Para elegir otra a mano, genera ese "
            "bloque con «modo_lote» desmarcado."
        )
        return None
    if eleccion == RECOMENDADA:
        numero = generacion.recomendada
    elif eleccion.isdecimal():
        numero = int(eleccion)
    else:
        unica = generacion.tomas[0]
        if cantidad > 1:
            print(
                f"👉 Escucha las {cantidad} tomas de la celda 6, elige una aquí (la recomendada es la "
                f"{generacion.recomendada}) y vuelve a ejecutar esta celda."
            )
        elif unica.en_drive is not None:
            print(f"ℹ️ La celda 6 generó una sola toma y ya está en {describir_ruta(unica.en_drive)}.")
        else:
            print("ℹ️ La celda 6 generó una sola toma; elige «1» si quieres guardarla en Drive.")
        return None
    if not 1 <= numero <= cantidad:
        raise ErrorUsuario(f"La última generación tiene {cantidad} toma(s): elige un número entre 1 y {cantidad}.")
    toma = generacion.tomas[numero - 1]
    print(f"🎯 Toma {numero} · semilla {toma.semilla}")
    try:
        guardar_toma_en_drive(toma, generacion)
    finally:  # aunque Drive falle, la descarga pedida se hace
        if descargar_archivo:
            descargar(toma)
    return toma


# ─── Ejecución ───
TOMA_GUARDADA = guardar_toma_elegida(toma_elegida, ULTIMA_GENERACION, descargar_archivo=descargar_tambien)

### Si algo no sale bien

| Problema | Qué hacer |
|---|---|
| «Este entorno no tiene GPU» | **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU** y empieza otra vez desde la celda 1. |
| Quieres cambiar de modelo (XTTS-v2 ↔ Chatterbox) | Cada uno instala versiones distintas de PyTorch: **Entorno de ejecución → Desconectar y eliminar entorno de ejecución**, elige el otro en la celda 1 y empieza de nuevo. |
| «mount failed» o Drive no se conecta | Ejecuta la celda otra vez y, en la ventana de Google, elige **la misma cuenta** con la que abriste Colab y marca **todos los permisos**. Si sigue: *Entorno de ejecución → Desconectar y eliminar entorno de ejecución* y empieza desde la celda 1, o usa el botón de Drive del panel *Archivos*. Mientras tanto, en la celda 4 elige «Subir desde mi computadora». |
| No encuentra tu audio | Escribe la ruta dentro de *Mi unidad*, p. ej. `Audios/mi voz.m4a`. |
| La voz no se parece | Usa 30–60 s de audio limpio, o una carpeta con varios audios tuyos. Activa `reducir_ruido` si se oye ruido de fondo. Para el máximo parecido, entrena el modelo con tu voz con el cuaderno [Entrenar XTTS con tu voz](https://colab.research.google.com/github/JoelPasapera/clonador-voz/blob/main/Entrenar_XTTS_con_tu_voz.ipynb). |
| Sonidos raros, repeticiones o balbuceos | Deja activo `control_de_calidad` y, si persiste, activa `verificar_con_whisper`. Pide 3 o 4 `tomas` y quédate con la recomendada (celda 7). También ayuda bajar `temperatura` (0.5–0.65). |
| «No encontré la voz …» en un diálogo | El nombre de `[voz: …]` debe ser una voz incluida (la celda 4 imprime la lista) o un perfil guardado con el **mismo modelo** que tienes cargado. |
| Lee mal una sigla, una abreviatura o un nombre | Agrega una regla en `pronunciacion` (celda 5): `DNI = de ene i`, `Jr. = jirón` (el modelo lee «Jr.» como «junior»), `Av. = avenida`. |
| Lee mal un precio o una cifra | Deja marcada `leer_montos_y_cifras`. Si aun así falla, escribe la cifra sin separadores (`1500`) o en letras. |
| Una palabra suelta en otro idioma suena rara | El modelo lee mejor frases completas: mete unas palabras más dentro de `[en]…[/en]` o pide varias tomas. |
| La música tapa la voz | Baja `volumen_musica` (p. ej. a −24). La música ya baja 12 dB más mientras hablas. |
| Necesitas más volumen para redes sociales | En `volumen` (celda 6) elige «Alto: -14 LUFS (redes)». Los picos nunca pasan de −1 dBFS. |
| Te gustó una toma y quieres repetirla | Copia su semilla en el campo `semilla`, deja `tomas` en 1 y vuelve a generar. |
| Textos muy largos (más de unos 20 minutos de audio) | Usa `modo_lote` y separa el texto en bloques con `---` (por ejemplo, uno por capítulo): cada bloque se guarda apenas termina. |
| La instalación falla tras una actualización de Colab | En la celda 1 agrega a `PAQUETES_EXTRA` la versión que indique el error (p. ej. `"transformers<5.19"`), usa **Entorno de ejecución → Reiniciar sesión** y vuelve a empezar. |

Los audios generados quedan en `/content/clonador_voz/salidas` (se borran al cerrar la sesión) y, si lo activaste, también en la carpeta de Drive que elegiste. Tus voces guardadas están en `Mi unidad/Clonador de voz/voces` y tus modelos entrenados en `Mi unidad/Clonador de voz/modelos`.